
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture — Feature Engineering in Unity Catalog

## Overview

This lecture covers the following three sections that build on each other:

- **A. Why a Feature Store?**. The architecture of Feature Engineering in Unity Catalog, its key functionalities (featurization and feature tables), and the three downstream workflows it supports: training dataset creation, batch scoring, and online serving
- **B. Feature Engineering in Unity Catalog**. How feature tables are Delta tables with primary keys, the unified permission model, unified lineage, integration with MLflow and Model Serving, point-in-time lookups, Lakebase and Online Feature Stores, and Feature Views for declaring features that Databricks computes for you, and how Feature Engineering in Unity Catalog replaces the legacy Workspace Feature Store
- **C. Conclusion**. Summary of key takeaways and how the components fit together

## Learning Objectives

By the end of this lecture, you will be able to:

- Explain what a feature store is and the four problems it solves (discovery, lineage, train/serve skew, and online serving)
- Distinguish Workspace Feature Store from Feature Engineering in Unity Catalog and identify the recommended path for new work
- Describe how feature tables are Delta tables in Unity Catalog with a primary key, governed like any other Unity Catalog table
- Explain how feature tables integrate with MLflow and Model Serving for batch and online inference
- Explain how point-in-time lookups on time series feature tables produce leakage-free training sets
- Explain what Lakebase is and why real-time inference needs it in addition to Delta feature tables
- Describe how Online Feature Stores (backed by Lakebase) serve features for real-time inference
- Distinguish feature tables from Feature Views, and identify when each is the better fit
- Describe how `FeatureEngineeringClient` is used to create, read, update, and drop a feature table (you do this hands-on in the 3.2 demo and 3.3 lab)

## A. Why a Feature Store?

### A1. What a Feature Actually Is

A **feature** is any value used as input to an ML model. For a churn model, features might be:

- Aggregations of raw data over time windows (e.g., *trailing 7-day purchases*)
- Joined combinations of datasets (e.g., *customer demographics joined to transactions*)
- Complex derived values (e.g., *estimated customer lifetime value*)

The process of producing these values from raw data is **feature engineering**. Earlier modules of this course covered the mechanics — loading data, imputing, encoding, scaling. This module covers what to do once you have features worth keeping: **store them in a way the rest of your team and your production models can use**.


### A2. What is a Feature Store?

A **feature store** manages features, or input data to a machine learning model. It lets us **store, retrieve, and share** features — ensuring that the same data used for model training is also used for inference. This reduces duplication of work and promotes collaboration across teams.

*An example of a recommendation system:*

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 16px 0;">

<!-- Left table: Raw Data -->
<div style="flex: 1; min-width: 220px;">
<div style="text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Raw Data</div>
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 10px 14px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Source</th>
      <th style="padding: 10px 14px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Examples</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">User Data</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">Age, location, signup date</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Product Data</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">Category, price, rating</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Transaction Data</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">Purchases, clicks, timestamps</td>
    </tr>
  </tbody>
</table>
</div>

<!-- Right table: Types of Features -->
<div style="flex: 1; min-width: 220px;">
<div style="text-align: center; font-weight: 700; color: #2272B4; margin-bottom: 10px; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase;">Types of Features</div>
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 10px 14px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Type</th>
      <th style="padding: 10px 14px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Example</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Transformations</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">Log price, age buckets</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Aggregations</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">7-day purchase count, avg rating</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Embeddings</strong></td>
      <td style="padding: 10px 14px; border-bottom: 1px solid #EEEDE9;">User vector, product vector</td>
    </tr>
  </tbody>
</table>
</div>

</div>

<!-- Flow SVG: Raw Data → Feature Engineering → Feature Store → ML Model → Outcome (new style) -->
<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 160" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <defs>
    <marker id="dgb-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Raw Data -->
  <rect x="5" y="45" width="140" height="70" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="77" y="74" text-anchor="middle" font-size="15" font-weight="600" fill="#1B3139">Raw Data</text>
  <text x="77" y="93" text-anchor="middle" font-size="12" fill="#618794">Users, Products,</text>
  <text x="77" y="104" text-anchor="middle" font-size="12" fill="#618794">and Transactions</text>
  <path d="M145 77 L172 77" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#dgb-arrow)"/>
  <!-- Feature Engineering -->
  <rect x="172" y="45" width="180" height="70" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="263" y="74" text-anchor="middle" font-size="15" font-weight="600" fill="#2272B4">Feature Engineering</text>
  <text x="263" y="93" text-anchor="middle" font-size="12" fill="#618794">Transforms, Aggregates,</text>
  <text x="263" y="104" text-anchor="middle" font-size="12" fill="#618794">and Embeddings</text>
  <path d="M352 77 L376 77" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#dgb-arrow)"/>
  <!-- Feature Store -->
  <rect x="376" y="45" width="160" height="75" rx="8" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="458" y="74" text-anchor="middle" font-size="15" font-weight="600" fill="#FF5F46">Feature Store</text>
  <text x="458" y="93" text-anchor="middle" font-size="12" fill="#618794">Governed and</text>
  <text x="458" y="104" text-anchor="middle" font-size="12" fill="#618794">Discoverable Features</text>
  <path d="M536 77 L563 77" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#dgb-arrow)"/>
  <!-- ML Model -->
  <rect x="563" y="45" width="125" height="70" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <text x="625" y="74" text-anchor="middle" font-size="15" font-weight="600" fill="#1B3139">ML Model</text>
  <text x="625" y="93" text-anchor="middle" font-size="12" fill="#618794">Training & Inference</text>
  <path d="M688 77 L725 77" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#dgb-arrow)"/>
  <!-- Outcome -->
  <rect x="725" y="45" width="120" height="70" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="785" y="74" text-anchor="middle" font-size="15" font-weight="600" fill="#00A972">Outcome</text>
  <text x="785" y="93" text-anchor="middle" font-size="12" fill="#618794">Recommendations</text>
</svg>
</div>

### A3. Four Problems a Feature Store Solves

Without a feature store, every team rebuilds the same features independently, leading to divergent definitions and silent production issues. A feature store gives you **one shared source of truth** for features — discoverable, governed, and reusable across batch and online.

<br>
<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 280px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 4px 16px rgba(27,49,57,0.13); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.green{--accent:#00A972;} .coral{--accent:#FF5F46;}
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 13pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 12px; }
.ds-card li:last-child { margin-bottom: 0; }
.dd-head { font-size: 18pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
</style>

<div class="ds-row">
  <div class="ds-card coral">
    <div class="dd-head">Without a Feature Store</div>
    <ul>
      <li><strong style="color:#2272B4;">Discovery:</strong> Multiple teams invent the same feature with different definitions. No one knows what already exists.</li>
      <li><strong style="color:#2272B4;">Lineage:</strong> At audit time, no one can answer <em>"how was this feature computed, and from what data?"</em></li>
      <li><strong style="color:#2272B4;">Skew:</strong> Training and serving features are computed by different pipelines — small differences cause silent model degradation in production.</li>
      <li><strong style="color:#2272B4;">Online Serving:</strong> Features computed only inside a training notebook are hard to reuse consistently for batch or real-time inference.</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="dd-head">With a Feature Store</div>
    <ul>
      <li><strong style="color:#2272B4;">Discovery:</strong> A searchable catalog lets teams find and reuse existing features instead of rebuilding them.</li>
      <li><strong style="color:#2272B4;">Lineage:</strong> Automatically tracks data sources, notebooks, jobs, and models associated with each feature — essential for audits.</li>
      <li><strong style="color:#2272B4;">Skew:</strong> A single source of truth — the same feature definitions are used for training and inference, which reduces divergence when keys, timestamps, and the serving workflow are set up correctly.</li>
      <li><strong style="color:#2272B4;">Online Serving:</strong> Publishes feature tables to a low-latency online store for real-time lookup, without rewriting logic.</li>
    </ul>
  </div>
</div>
</div>
<br>



<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">The Databricks feature store is now called <strong>Feature Engineering in Unity Catalog</strong>; some older documentation and UI pages still say <strong>Databricks Feature Store</strong>. The legacy <strong>Workspace Feature Store</strong> is deprecated and accessed via <code>FeatureStoreClient</code> from <code>databricks.feature_store</code>. For new work, use the Unity Catalog–backed feature store with <code>FeatureEngineeringClient</code> from <code>databricks.feature_engineering</code>.</div></div>


### A4. Feature Store Architecture on Databricks

Feature Engineering in Unity Catalog provides an end-to-end platform for defining, storing, and serving features. It has two core capabilities — **Featurization** and **Feature Tables** — and supports three downstream workflows: training dataset creation, batch scoring, and online serving.

---

#### Key Functionalities

<div style="display: flex; gap: 32px; flex-wrap: wrap; margin: 24px 0;">

<!-- Featurization -->
<div style="flex: 1; min-width: 300px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Featurization</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 30%;"><strong style="color: #2272B4;">Languages</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Python, SQL, Scala, R</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Purpose</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Define reusable and shareable feature logic to ensure consistency and efficiency in feature engineering</td>
    </tr>
  </tbody>
</table>
</div>

<!-- Feature Tables -->
<div style="flex: 1; min-width: 300px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Feature Tables</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 30%;"><strong style="color: #2272B4;">Built on</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Delta Lake — ACID transactions, versioning, and performance optimizations</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Organization</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Features organized logically into tables (e.g., Customer Features, Item Features) to simplify management and access</td>
    </tr>
  </tbody>
</table>
</div>

</div>


#### A4.1 Workflow Overview

<!-- Architecture SVG -->
<div style="max-width: 900px; margin: 24px auto;">
<svg width="100%" viewBox="0 0 630 275" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Feature Engineering in Unity Catalog Architecture</title>
  <defs>
    <marker id="a4-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Left: Featurization -->
  <rect x="10" y="68" width="140" height="100" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="80" y="106" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Featurization</text>
  <text x="80" y="127" text-anchor="middle" font-size="12" fill="#618794">Python · SQL · Scala · R</text>
  <text x="80" y="145" text-anchor="middle" font-size="12" fill="#618794">Reusable feature logic</text>
  <!-- Arrow: Featurization → Feature Tables -->
  <path d="M150 118 L193 118" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#a4-arrow)"/>
  <!-- Center: Feature Tables -->
  <rect x="198" y="48" width="180" height="140" rx="12" fill="#FF5F46" fill-opacity="0.08" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="288" y="82" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Feature Tables</text>
  <text x="288" y="103" text-anchor="middle" font-size="14" fill="#618794">Delta Lake</text>
  <line x1="228" y1="113" x2="348" y2="113" stroke="#EEEDE9" stroke-width="1"/>
  <text x="288" y="133" text-anchor="middle" font-size="12" fill="#618794">Customer Features</text>
  <text x="288" y="151" text-anchor="middle" font-size="12" fill="#618794">Item Features</text>
  <text x="288" y="169" text-anchor="middle" font-size="12" fill="#618794">Transaction Features</text>
  <!-- Arrows: Feature Tables → workflows -->
  <path d="M378 88 L425 41" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a4-arrow)"/>
  <path d="M378 118 L425 118" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a4-arrow)"/>
  <path d="M378 150 L425 195" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a4-arrow)"/>
  <!-- Training Dataset Creation -->
  <rect x="433" y="12" width="185" height="58" rx="8" fill="#2272B4" fill-opacity="0.10" stroke="#2272B4" stroke-width="1.5"/>
  <text x="525" y="36" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Training Dataset</text>
  <text x="525" y="56" text-anchor="middle" font-size="12" fill="#618794">Snapshot features → MLflow</text>
  <!-- Batch Scoring -->
  <rect x="433" y="89" width="185" height="58" rx="8" fill="#4299E0" fill-opacity="0.10" stroke="#4299E0" stroke-width="1.5"/>
  <text x="525" y="113" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Batch Scoring</text>
  <text x="525" y="133" text-anchor="middle" font-size="12" fill="#618794">Scheduled predictions</text>
  <!-- Online Serving -->
  <rect x="433" y="166" width="185" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="525" y="190" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Online Serving</text>
  <text x="525" y="210" text-anchor="middle" font-size="12" fill="#618794">Low-latency feature lookup</text>
  <!-- Bottom: summary bar -->
  <rect x="80" y="240" width="470" height="30" rx="6" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <text x="315" y="259" text-anchor="middle" font-size="12" fill="#0b2026">One platform: define features once → use everywhere (training, batch, real-time)</text>
</svg>
</div>


#### A4.2 Workflow Details

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Workflow</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Process</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Use Case</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Training Dataset Creation</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Features are snapshot and packaged into datasets, ready for training with tools like MLflow. Supports on-demand feature computation at query time.</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Model development &amp; experimentation</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Batch Scoring</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Features are loaded and utilized for scoring in batch processing scenarios. The same feature definitions used in training are applied at inference time.</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Scheduled predictions, reporting pipelines</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Online Serving</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Publish features for real-time model inference. Feature tables are synced to low-latency stores for dynamic, responsive ML applications.</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Fraud detection, recommendations, loan approval</td></tr></tbody></table></div>

## B. Feature Engineering in Unity Catalog

### B1. A Feature Table Is Just a Delta Table with Rules

On Databricks, a **feature table** is a regular Delta table in Unity Catalog that satisfies two requirements:

1. A **primary key** column (or columns) with a `PRIMARY KEY` constraint
2. The primary key column(s) must be `NOT NULL`

Any Delta table in Unity Catalog with a primary key constraint can be used as a feature table for model training and inference. You can also promote an existing Delta table by adding a `PRIMARY KEY` constraint — no registration step.

---

#### Feature Table Structure

<div style="max-width: 1000px; margin: 32px auto;">
<svg width="100%" viewBox="0 0 980 340" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Feature Table as a UC Delta Table</title>
  <defs>
    <linearGradient id="b1-grad" x1="0%" y1="0%" x2="0%" y2="100%">
      <stop offset="0%" style="stop-color:#2272B4;stop-opacity:0.08"/>
      <stop offset="100%" style="stop-color:#2272B4;stop-opacity:0.02"/>
    </linearGradient>
  </defs>
  <!-- Three-level namespace header -->
  <rect x="30" y="15" width="660" height="36" rx="6" fill="#1B3139"/>
  <text x="350" y="38" text-anchor="middle" font-size="13" font-weight="700" fill="#e3f2fd">catalog.schema.customer_features</text>
  <text x="720" y="38" font-size="13" fill="#FF5F46" font-weight="600">← UC three-level namespace</text>
  <!-- Main table box -->
  <rect x="30" y="55" width="660" height="200" rx="8" fill="url(#b1-grad)" stroke="#2272B4" stroke-width="1.5"/>
  <!-- Column rows -->
  <rect x="50" y="68" width="620" height="28" rx="4" fill="#FF5F46" fill-opacity="0.08" stroke="#FF5F46" stroke-width="1"/>
  <text x="65" y="87" font-size="14" font-weight="700" fill="#0b2026">customerID</text>
  <text x="190" y="87" font-size="12" fill="#FF5F46" font-weight="600">PK, NOT NULL</text>
  <text x="720" y="87" font-size="13" fill="#FF5F46" font-weight="600">← primary key requirement</text>
  <rect x="50" y="102" width="620" height="26" rx="4" fill="#ffffff" stroke="#EEEDE9" stroke-width="1"/>
  <text x="65" y="120" font-size="14" fill="#0b2026">tenure</text>
  <text x="190" y="120" font-size="12" fill="#1B3139">INT</text>
  <rect x="50" y="132" width="620" height="26" rx="4" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <text x="65" y="150" font-size="14" fill="#0b2026">MonthlyCharges</text>
  <text x="190" y="150" font-size="12" fill="#1B3139">DOUBLE</text>
  <rect x="50" y="162" width="620" height="26" rx="4" fill="#ffffff" stroke="#EEEDE9" stroke-width="1"/>
  <text x="65" y="180" font-size="14" fill="#0b2026">TotalCharges</text>
  <text x="190" y="180" font-size="12" fill="#1B3139">DOUBLE</text>
  <rect x="50" y="192" width="620" height="26" rx="4" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <text x="65" y="210" font-size="14" fill="#0b2026">tenure_group</text>
  <text x="190" y="210" font-size="12" fill="#1B3139">STRING</text>
  <text x="720" y="210" font-size="13" fill="#2272B4" font-weight="600">← derived feature</text>
  <!-- Bottom layers -->
  <rect x="30" y="232" width="660" height="22" rx="0" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1"/>
  <text x="350" y="247" text-anchor="middle" font-size="12" font-weight="600" fill="#00805A">Underneath: Delta Lake</text>
  <!-- Top capabilities bar -->
  <rect x="60" y="270" width="600" height="50" rx="8" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <text x="350" y="290" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">On top: Unity Catalog governance</text>
  <text x="350" y="308" text-anchor="middle" font-size="12" fill="#1B3139">Security · Lineage · Discovery · ACLs · Time Travel · Tagging · Cross-workspace Access</text>
</svg>
</div>


#### B1.1 Feature Tables = UC Tables + Metadata

Feature tables behave like regular Unity Catalog tables but include additional metadata for feature management. Feature Engineering integrates fully with Unity Catalog, meaning any Unity Catalog Delta table with a primary key can be used as a feature table.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Feature Table Property</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Unity Catalog Equivalent</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Benefit</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Feature table description</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Table comment (<code>COMMENT ON TABLE</code>)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Discoverable in Catalog Explorer search</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Feature table schema</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Table schema (columns &amp; types)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Schema evolution, enforcement, versioning</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;">Namespace</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Three-level: <code>catalog.schema.table</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Consistent cross-platform access</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Access control</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">UC permissions (<code>GRANT</code> / <code>REVOKE</code>)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Single permission model, cross-workspace</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Lineage &amp; auditing</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">UC lineage graph + audit logs</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Trace which jobs, models, and dashboards use each feature</td></tr></tbody></table></div>

<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 4px 12px rgba(27,49,57,0.08); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaway</div>
  <div class="ds-callout-title">Any Delta table in Unity Catalog with a primary key can be used as a feature table.</div>
  <ul>
    <li>No registration step required—just add a <code>PRIMARY KEY</code> constraint.</li>
    <li>Existing Delta tables can be promoted to feature tables by adding a primary key.</li>
    <li>Feature tables inherit all UC capabilities: governance, lineage, permissions, and cross-workspace access.</li>
  </ul>
</div>
</div>


### B2. Unified Permission Model

**Purpose:** Establish a consistent and secure management system for data access across feature tables and all other Unity Catalog assets — ensuring that governance policies are uniformly applied.

---

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 24px 0;">

<!-- Left: Key points -->
<div style="flex: 1; min-width: 320px;">

<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Secure features with built-in governance</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 28%;"><strong style="color: #2272B4;">Permission Model</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Feature data and metadata are governed by the Unity Catalog permission model — the same security and governance rules used for other UC tables apply to feature tables</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Centralized Control</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Permissions managed across catalogs, schemas, tables, and views from a central platform — applicable across all workspaces</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Future-Proof</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Any future improvements to data governance in Unity Catalog will automatically extend to feature data</td>
    </tr>
  </tbody>
</table>

</div>
</div>


#### B2.1 Example of Permission Grants on a Feature Table

<div style="max-width: 780px; margin: 24px auto;">
<svg width="100%" viewBox="0 0 760 330" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Unified Permission Model — UC Grants on Feature Tables</title>
  <!-- Background -->
  <rect x="0" y="0" width="760" height="330" rx="10" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1"/>
  <!-- Title bar -->
  <rect x="0" y="0" width="760" height="40" rx="10" fill="#EBF3FA"/>
  <rect x="0" y="20" width="760" height="20" fill="#EBF3FA"/>
  <text x="24" y="26" font-size="13" font-weight="700" fill="#0b2026">Permissions — catalog.ml_features.customer_features</text>
  <text x="580" y="26" font-size="10" fill="#1B3139">Unity Catalog</text>
  <!-- Table header row -->
  <rect x="20" y="52" width="720" height="30" rx="4" fill="#ffffff" stroke="#EEEDE9" stroke-width="1"/>
  <text x="40" y="72" font-size="11" font-weight="700" fill="#0b2026">Principal</text>
  <text x="220" y="72" font-size="11" font-weight="700" fill="#0b2026">Type</text>
  <text x="340" y="72" font-size="11" font-weight="700" fill="#0b2026">Privilege</text>
  <text x="500" y="72" font-size="11" font-weight="700" fill="#0b2026">Inherited From</text>
  <text x="660" y="72" font-size="11" font-weight="700" fill="#0b2026">Status</text>
  <!-- Row 1: data_scientists group -->
  <rect x="20" y="84" width="720" height="28" fill="#ffffff"/>
  <line x1="20" y1="112" x2="740" y2="112" stroke="#EEEDE9" stroke-width="1"/>
  <circle cx="36" cy="98" r="7" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="1"/>
  <text x="50" y="102" font-size="10" fill="#0b2026">data_scientists</text>
  <text x="220" y="102" font-size="10" fill="#1B3139">Group</text>
  <rect x="340" y="90" width="60" height="18" rx="3" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="0.8"/>
  <text x="352" y="103" font-size="10" font-weight="600" fill="#00805A">SELECT</text>
  <text x="500" y="102" font-size="11" fill="#1B3139">catalog.ml_features</text>
  <circle cx="676" cy="98" r="5" fill="#00A972"/>
  <text x="686" y="102" font-size="10" fill="#00805A">Active</text>
  <!-- Row 2: ml_engineers group -->
  <rect x="20" y="112" width="720" height="28" fill="#F9F7F4"/>
  <line x1="20" y1="140" x2="740" y2="140" stroke="#EEEDE9" stroke-width="1"/>
  <circle cx="36" cy="126" r="7" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="1"/>
  <text x="50" y="130" font-size="10" fill="#0b2026">ml_engineers</text>
  <text x="220" y="130" font-size="10" fill="#1B3139">Group</text>
  <rect x="340" y="118" width="60" height="18" rx="3" fill="#FF5F46" fill-opacity="0.10" stroke="#FF5F46" stroke-width="0.8"/>
  <text x="350" y="131" font-size="10" font-weight="600" fill="#98102A">MODIFY</text>
  <text x="500" y="130" font-size="10" fill="#1B3139">Direct grant</text>
  <circle cx="676" cy="126" r="5" fill="#00A972"/>
  <text x="686" y="130" font-size="10" fill="#00805A">Active</text>
  <!-- Row 3: ml_engineers SELECT -->
  <rect x="20" y="140" width="720" height="28" fill="#ffffff"/>
  <line x1="20" y1="168" x2="740" y2="168" stroke="#EEEDE9" stroke-width="1"/>
  <circle cx="36" cy="154" r="7" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="1"/>
  <text x="50" y="158" font-size="10" fill="#0b2026">ml_engineers</text>
  <text x="220" y="158" font-size="10" fill="#1B3139">Group</text>
  <rect x="340" y="146" width="60" height="18" rx="3" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="0.8"/>
  <text x="352" y="159" font-size="10" font-weight="600" fill="#00805A">SELECT</text>
  <text x="500" y="158" font-size="10" fill="#1B3139">catalog.ml_features</text>
  <circle cx="676" cy="154" r="5" fill="#00A972"/>
  <text x="686" y="158" font-size="10" fill="#00805A">Active</text>
  <!-- Row 4: pipeline_svc -->
  <rect x="20" y="168" width="720" height="28" fill="#F9F7F4"/>
  <line x1="20" y1="196" x2="740" y2="196" stroke="#EEEDE9" stroke-width="1"/>
  <circle cx="36" cy="182" r="7" fill="#FF5F46" fill-opacity="0.15" stroke="#FF5F46" stroke-width="1"/>
  <text x="50" y="186" font-size="10" fill="#0b2026">pipeline_svc_principal</text>
  <text x="220" y="186" font-size="10" fill="#1B3139">Service Principal</text>
  <rect x="340" y="174" width="60" height="18" rx="3" fill="#FF5F46" fill-opacity="0.10" stroke="#FF5F46" stroke-width="0.8"/>
  <text x="350" y="187" font-size="10" font-weight="600" fill="#98102A">MODIFY</text>
  <text x="500" y="186" font-size="10" fill="#1B3139">Direct grant</text>
  <circle cx="676" cy="182" r="5" fill="#00A972"/>
  <text x="686" y="186" font-size="10" fill="#00805A">Active</text>
  <!-- Row 5: analysts -->
  <rect x="20" y="196" width="720" height="28" fill="#ffffff"/>
  <line x1="20" y1="224" x2="740" y2="224" stroke="#EEEDE9" stroke-width="1"/>
  <circle cx="36" cy="210" r="7" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="1"/>
  <text x="50" y="214" font-size="10" fill="#0b2026">analysts</text>
  <text x="220" y="214" font-size="10" fill="#1B3139">Group</text>
  <rect x="340" y="202" width="60" height="18" rx="3" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="0.8"/>
  <text x="352" y="215" font-size="10" font-weight="600" fill="#00805A">SELECT</text>
  <text x="500" y="214" font-size="10" fill="#1B3139">catalog.ml_features</text>
  <circle cx="676" cy="210" r="5" fill="#00A972"/>
  <text x="686" y="214" font-size="10" fill="#00805A">Active</text>
  <!-- Separator -->
  <line x1="20" y1="234" x2="740" y2="234" stroke="#2272B4" stroke-width="0.5" stroke-dasharray="4,3"/>
  <!-- Bottom annotation -->
  <rect x="40" y="248" width="680" height="66" rx="8" fill="#2272B4" fill-opacity="0.05" stroke="#2272B4" stroke-width="1"/>
  <text x="380" y="270" text-anchor="middle" font-size="12" font-weight="700" fill="#0b2026">How it works</text>
  <text x="380" y="288" text-anchor="middle" font-size="11" fill="#1B3139">GRANT SELECT ON TABLE catalog.ml_features.customer_features TO `data_scientists`;</text>
  <text x="380" y="304" text-anchor="middle" font-size="11" fill="#1B3139">GRANT MODIFY ON TABLE catalog.ml_features.customer_features TO `ml_engineers`;</text>
</svg>
</div>

<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 4px 12px rgba(27,49,57,0.08); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaway</div>
  <div class="ds-callout-title">Feature table permissions are managed with Unity Catalog's unified permission model.</div>
  <ul>
    <li>Access to feature tables is controlled using standard UC <code>GRANT</code> and <code>REVOKE</code> SQL.</li>
    <li>Privileges like <code>SELECT</code> and <code>MODIFY</code> can be assigned to users, groups, and service principals at the catalog, schema, or table level.</li>
    <li>To read a feature table, a user also needs <code>USE CATALOG</code> on its catalog and <code>USE SCHEMA</code> on its schema, as for any Unity Catalog table.</li>
    <li>Feature data and metadata governance is consistent with all other UC assets—no special rules or exceptions.</li>
  </ul>
</div>
</div>


### B3. Unified Data Lineage

**Feature lineage and Unity Catalog lineage are one and the same.** Lineage information — the data flow and dependencies between tables, views, notebooks, jobs, and dashboards — is combined into a single unified view.

---

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 24px 0;">

<!-- Key points -->
<div style="flex: 1; min-width: 320px;">

<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What unified lineage provides</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 28%;"><strong style="color: #2272B4;">Single View</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Feature tables, their source data, and downstream usage are all tracked in one place within UC</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Full Traceability</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Trace how data flows — from raw inputs through transformations to final feature tables and models</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Dependency Insight</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Understand dependencies between assets to ensure consistency across the workflow</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Asset Coverage</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">Tables, views, notebooks, jobs, dashboards, and ML models — all connected in one lineage graph</td>
    </tr>
  </tbody>
</table>

</div>
</div>

<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 4px 12px rgba(27,49,57,0.08); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 0; padding-left: 20px; font-size: 12pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info    { --accent:#1B5162; --tint:#F8F9FC; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.danger  { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.success { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-callout-title { font-size: 14pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Key Takeaway</div>
  <div class="ds-callout-title">Feature lineage is fully integrated with Unity Catalog lineage.</div>
  <ul>
    <li>Visualize the full data flow: from raw inputs, through transformation notebooks, into feature tables, and out to models, dashboards, and scoring pipelines.</li>
    <li>All lineage is accessible from a single lineage explorer in Unity Catalog.</li>
    <li>Traceability and dependency insight across tables, notebooks, jobs, and ML models.</li>
  </ul>
</div>
</div>

#### B3.1 Lineage Graph: From Raw Data to Model Inference

The unified lineage view traces data flow across four stages — from raw sources through transformation notebooks into feature tables and finally to downstream consumers.

---

<div style="max-width: 860px; margin: 24px auto;">

<!-- Full flow diagram with separate grouped SVGs connected by arrows -->
<div style="display: flex; align-items: center; gap: 0; flex-wrap: nowrap; overflow-x: auto; padding: 16px 0;">

<!-- ======== STAGE 1: RAW SOURCES ======== -->
<div style="flex: 0 0 155px;">
<svg width="155" height="260" viewBox="0 0 155 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="77" y="16" text-anchor="middle" font-size="9" font-weight="700" fill="#2272B4">RAW SOURCES</text>
  <line x1="20" y1="24" x2="135" y2="24" stroke="#2272B4" stroke-width="1" opacity="0.3"/>
  <!-- raw_customers -->
  <rect x="10" y="40" width="130" height="52" rx="7" fill="#ffffff" stroke="#2272B4" stroke-width="1.5"/>
  <rect x="10" y="40" width="130" height="16" rx="7" fill="#2272B4" fill-opacity="0.08"/>
  <rect x="10" y="50" width="130" height="6" fill="#2272B4" fill-opacity="0.08"/>
  <circle cx="24" cy="49" r="4" fill="#2272B4" fill-opacity="0.4"/>
  <text x="77" y="51" text-anchor="middle" font-size="7" font-weight="600" fill="#2272B4">TABLE</text>
  <text x="77" y="72" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">raw_customers</text>
  <text x="77" y="85" text-anchor="middle" font-size="8" fill="#618794">demographics, tenure</text>
  <!-- raw_transactions -->
  <rect x="10" y="108" width="130" height="52" rx="7" fill="#ffffff" stroke="#2272B4" stroke-width="1.5"/>
  <rect x="10" y="108" width="130" height="16" rx="7" fill="#2272B4" fill-opacity="0.08"/>
  <rect x="10" y="118" width="130" height="6" fill="#2272B4" fill-opacity="0.08"/>
  <circle cx="24" cy="117" r="4" fill="#2272B4" fill-opacity="0.4"/>
  <text x="77" y="119" text-anchor="middle" font-size="7" font-weight="600" fill="#2272B4">TABLE</text>
  <text x="77" y="140" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">raw_transactions</text>
  <text x="77" y="153" text-anchor="middle" font-size="8" fill="#618794">charges, payments</text>
  <!-- raw_items -->
  <rect x="10" y="176" width="130" height="52" rx="7" fill="#ffffff" stroke="#2272B4" stroke-width="1.5"/>
  <rect x="10" y="176" width="130" height="16" rx="7" fill="#2272B4" fill-opacity="0.08"/>
  <rect x="10" y="186" width="130" height="6" fill="#2272B4" fill-opacity="0.08"/>
  <circle cx="24" cy="185" r="4" fill="#2272B4" fill-opacity="0.4"/>
  <text x="77" y="187" text-anchor="middle" font-size="7" font-weight="600" fill="#2272B4">TABLE</text>
  <text x="77" y="208" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">raw_items</text>
  <text x="77" y="221" text-anchor="middle" font-size="8" fill="#618794">product catalog</text>
</svg>
</div>

<!-- ======== ARROW 1: Sources → Notebooks ======== -->
<div style="flex: 0 0 50px; display: flex; align-items: center; justify-content: center;">
<svg width="50" height="260" viewBox="0 0 50 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <defs>
    <marker id="arr1" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M1 1L9 5L1 9" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Multiple flow lines -->
  <line x1="5" y1="66" x2="42" y2="90" stroke="#2272B4" stroke-width="1.5" marker-end="url(#arr1)"/>
  <line x1="5" y1="134" x2="42" y2="110" stroke="#2272B4" stroke-width="1.5" marker-end="url(#arr1)"/>
  <line x1="5" y1="134" x2="42" y2="180" stroke="#2272B4" stroke-width="1.5" marker-end="url(#arr1)"/>
  <line x1="5" y1="202" x2="42" y2="190" stroke="#2272B4" stroke-width="1.5" marker-end="url(#arr1)"/>
</svg>
</div>

<!-- ======== STAGE 2: NOTEBOOKS ======== -->
<div style="flex: 0 0 155px;">
<svg width="155" height="260" viewBox="0 0 155 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="77" y="16" text-anchor="middle" font-size="9" font-weight="700" fill="#FF5F46">NOTEBOOKS</text>
  <line x1="20" y1="24" x2="135" y2="24" stroke="#FF5F46" stroke-width="1" opacity="0.3"/>
  <!-- feature_engineering -->
  <rect x="10" y="56" width="130" height="68" rx="7" fill="#ffffff" stroke="#FF5F46" stroke-width="1.5"/>
  <rect x="10" y="56" width="130" height="16" rx="7" fill="#FF5F46" fill-opacity="0.06"/>
  <rect x="10" y="66" width="130" height="6" fill="#FF5F46" fill-opacity="0.06"/>
  <circle cx="24" cy="65" r="4" fill="#FF5F46" fill-opacity="0.4"/>
  <text x="77" y="67" text-anchor="middle" font-size="7" font-weight="600" fill="#FF5F46">NOTEBOOK</text>
  <text x="77" y="88" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">feature_engineering</text>
  <text x="77" y="103" text-anchor="middle" font-size="8" fill="#618794">compute customer &amp;</text>
  <text x="77" y="115" text-anchor="middle" font-size="8" fill="#618794">transaction features</text>
  <!-- item_feature_pipe -->
  <rect x="10" y="148" width="130" height="68" rx="7" fill="#ffffff" stroke="#FF5F46" stroke-width="1.5"/>
  <rect x="10" y="148" width="130" height="16" rx="7" fill="#FF5F46" fill-opacity="0.06"/>
  <rect x="10" y="158" width="130" height="6" fill="#FF5F46" fill-opacity="0.06"/>
  <circle cx="24" cy="157" r="4" fill="#FF5F46" fill-opacity="0.4"/>
  <text x="77" y="159" text-anchor="middle" font-size="7" font-weight="600" fill="#FF5F46">NOTEBOOK</text>
  <text x="77" y="180" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">item_feature_pipe</text>
  <text x="77" y="195" text-anchor="middle" font-size="8" fill="#618794">item enrichment &amp;</text>
  <text x="77" y="207" text-anchor="middle" font-size="8" fill="#618794">catalog features</text>
</svg>
</div>

<!-- ======== ARROW 2: Notebooks → Feature Tables ======== -->
<div style="flex: 0 0 50px; display: flex; align-items: center; justify-content: center;">
<svg width="50" height="260" viewBox="0 0 50 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <defs>
    <marker id="arr2" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M1 1L9 5L1 9" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <line x1="5" y1="80" x2="42" y2="66" stroke="#FF5F46" stroke-width="1.5" marker-end="url(#arr2)"/>
  <line x1="5" y1="100" x2="42" y2="134" stroke="#FF5F46" stroke-width="1.5" marker-end="url(#arr2)"/>
  <line x1="5" y1="182" x2="42" y2="202" stroke="#FF5F46" stroke-width="1.5" marker-end="url(#arr2)"/>
</svg>
</div>

<!-- ======== STAGE 3: FEATURE TABLES ======== -->
<div style="flex: 0 0 155px;">
<svg width="155" height="260" viewBox="0 0 155 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="77" y="16" text-anchor="middle" font-size="9" font-weight="700" fill="#00A972">FEATURE TABLES</text>
  <line x1="20" y1="24" x2="135" y2="24" stroke="#00A972" stroke-width="1" opacity="0.3"/>
  <!-- customer_features -->
  <rect x="10" y="40" width="130" height="52" rx="7" fill="#ffffff" stroke="#00A972" stroke-width="1.8"/>
  <rect x="10" y="40" width="130" height="16" rx="7" fill="#00A972" fill-opacity="0.08"/>
  <rect x="10" y="50" width="130" height="6" fill="#00A972" fill-opacity="0.08"/>
  <circle cx="24" cy="49" r="4" fill="#00A972" fill-opacity="0.4"/>
  <text x="77" y="51" text-anchor="middle" font-size="7" font-weight="600" fill="#00A972">FEATURE TABLE</text>
  <text x="77" y="72" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">customer_features</text>
  <text x="77" y="85" text-anchor="middle" font-size="8" fill="#618794">PK: customerID</text>
  <!-- txn_features -->
  <rect x="10" y="108" width="130" height="52" rx="7" fill="#ffffff" stroke="#00A972" stroke-width="1.8"/>
  <rect x="10" y="108" width="130" height="16" rx="7" fill="#00A972" fill-opacity="0.08"/>
  <rect x="10" y="118" width="130" height="6" fill="#00A972" fill-opacity="0.08"/>
  <circle cx="24" cy="117" r="4" fill="#00A972" fill-opacity="0.4"/>
  <text x="77" y="119" text-anchor="middle" font-size="7" font-weight="600" fill="#00A972">FEATURE TABLE</text>
  <text x="77" y="140" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">txn_features</text>
  <text x="77" y="153" text-anchor="middle" font-size="8" fill="#618794">PK: txn_id</text>
  <!-- item_features -->
  <rect x="10" y="176" width="130" height="52" rx="7" fill="#ffffff" stroke="#00A972" stroke-width="1.8"/>
  <rect x="10" y="176" width="130" height="16" rx="7" fill="#00A972" fill-opacity="0.08"/>
  <rect x="10" y="186" width="130" height="6" fill="#00A972" fill-opacity="0.08"/>
  <circle cx="24" cy="185" r="4" fill="#00A972" fill-opacity="0.4"/>
  <text x="77" y="187" text-anchor="middle" font-size="7" font-weight="600" fill="#00A972">FEATURE TABLE</text>
  <text x="77" y="208" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">item_features</text>
  <text x="77" y="221" text-anchor="middle" font-size="8" fill="#618794">PK: item_id</text>
</svg>
</div>

<!-- ======== ARROW 3: Feature Tables → Downstream ======== -->
<div style="flex: 0 0 50px; display: flex; align-items: center; justify-content: center;">
<svg width="50" height="260" viewBox="0 0 50 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <defs>
    <marker id="arr3" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M1 1L9 5L1 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <line x1="5" y1="66" x2="42" y2="66" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="4,2" marker-end="url(#arr3)"/>
  <line x1="5" y1="134" x2="42" y2="134" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="4,2" marker-end="url(#arr3)"/>
  <line x1="5" y1="202" x2="42" y2="202" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="4,2" marker-end="url(#arr3)"/>
  <!-- cross-connections -->
  <line x1="5" y1="66" x2="42" y2="134" stroke="#1B3139" stroke-width="1" stroke-dasharray="3,3" opacity="0.35" marker-end="url(#arr3)"/>
  <line x1="5" y1="134" x2="42" y2="202" stroke="#1B3139" stroke-width="1" stroke-dasharray="3,3" opacity="0.35" marker-end="url(#arr3)"/>
</svg>
</div>

<!-- ======== STAGE 4: DOWNSTREAM ======== -->
<div style="flex: 0 0 155px;">
<svg width="155" height="260" viewBox="0 0 155 260" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <text x="77" y="16" text-anchor="middle" font-size="9" font-weight="700" fill="#1B3139">DOWNSTREAM</text>
  <line x1="20" y1="24" x2="135" y2="24" stroke="#1B3139" stroke-width="1" opacity="0.3"/>
  <!-- ML Model -->
  <rect x="10" y="40" width="130" height="52" rx="7" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="10" y="40" width="130" height="16" rx="7" fill="#1B3139" fill-opacity="0.06"/>
  <rect x="10" y="50" width="130" height="6" fill="#1B3139" fill-opacity="0.06"/>
  <circle cx="24" cy="49" r="4" fill="#1B3139" fill-opacity="0.3"/>
  <text x="77" y="51" text-anchor="middle" font-size="7" font-weight="600" fill="#1B3139">MODEL</text>
  <text x="77" y="72" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">churn_predictor</text>
  <text x="77" y="85" text-anchor="middle" font-size="8" fill="#618794">v3 · Production</text>
  <!-- Dashboard -->
  <rect x="10" y="108" width="130" height="52" rx="7" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="10" y="108" width="130" height="16" rx="7" fill="#1B3139" fill-opacity="0.06"/>
  <rect x="10" y="118" width="130" height="6" fill="#1B3139" fill-opacity="0.06"/>
  <circle cx="24" cy="117" r="4" fill="#1B3139" fill-opacity="0.3"/>
  <text x="77" y="119" text-anchor="middle" font-size="7" font-weight="600" fill="#1B3139">DASHBOARD</text>
  <text x="77" y="140" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">Feature Monitoring</text>
  <text x="77" y="153" text-anchor="middle" font-size="8" fill="#618794">drift &amp; freshness</text>
  <!-- Workflow -->
  <rect x="10" y="176" width="130" height="52" rx="7" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="10" y="176" width="130" height="16" rx="7" fill="#1B3139" fill-opacity="0.06"/>
  <rect x="10" y="186" width="130" height="6" fill="#1B3139" fill-opacity="0.06"/>
  <circle cx="24" cy="185" r="4" fill="#1B3139" fill-opacity="0.3"/>
  <text x="77" y="187" text-anchor="middle" font-size="7" font-weight="600" fill="#1B3139">JOB</text>
  <text x="77" y="208" text-anchor="middle" font-size="10" font-weight="600" fill="#1B3139">scoring_pipeline</text>
  <text x="77" y="221" text-anchor="middle" font-size="8" fill="#618794">batch inference</text>
</svg>
</div>

</div>

<!-- Legend -->
<div style="margin-top: 8px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 12px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <tr style="background-color: #ffffff;">
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;"><span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #2272B4; opacity: 0.6; vertical-align: middle;"></span> <strong style="color: #2272B4;">Source Table</strong></td>
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;"><span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #FF5F46; opacity: 0.6; vertical-align: middle;"></span> <strong style="color: #FF5F46;">Notebook</strong></td>
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;"><span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #00A972; opacity: 0.6; vertical-align: middle;"></span> <strong style="color: #00A972;">Feature Table</strong></td>
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;"><span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #1B3139; opacity: 0.6; vertical-align: middle;"></span> <strong style="color: #1B3139;">Downstream</strong></td>
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;">——→ <strong>Data flow</strong></td>
    <td style="padding: 8px 12px; text-align: center; border: 1px solid #EEEDE9;">- - -→ <strong>Consumption</strong></td>
  </tr>
</table>
</div>
</div>


### B4. Integration with MLflow and Model Serving

**Purpose:** Make model deployment easier. Feature Engineering in Unity Catalog integrates with both MLflow and Model Serving to support seamless model training, packaging, and inference.

---

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 24px 0;">

<!-- Integration with MLflow -->
<div style="flex: 1; min-width: 340px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Integration with MLflow</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 30%;"><strong style="color: #2272B4;">Metadata Packaging</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">When a model is trained on a training set from <code>create_training_set</code> and logged with <code>fe.log_model</code>, it is packaged with the feature lookup metadata it needs (which feature tables and keys to use). A model logged with plain MLflow does not carry this metadata</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Traceability</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">The model retains references to the features it used during training, helping trace and understand each feature's origin and use</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #2272B4;">Inference Time</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">At inference time, the model can look up required feature values from the feature tables at runtime — the request must include the primary key columns (e.g., <code>user_id</code>)</td>
    </tr>
  </tbody>
</table>
</div>

<!-- Integration with Model Serving -->
<div style="flex: 1; min-width: 340px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th colspan="2" style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #00A972;">Integration with Model Serving</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9; width: 30%;"><strong style="color: #2272B4;">Auto-Retrieval</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">When a model logged with <code>fe.log_model</code> is deployed for batch or online inference, it retrieves the needed features from the feature tables (for online inference, the tables must be published to an Online Feature Store)</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #00A972;">Batch Mode</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong>High throughput</strong> — features retrieved from the offline store and joined with new data before scoring</td>
    </tr>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong style="color: #00A972;">Online Inference</strong></td>
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;"><strong>Low latency</strong> — features retrieved from the online store in real-time for responsive predictions</td>
    </tr>
  </tbody>
</table>
</div>

</div>

---

#### How It Works Together

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Phase</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What Happens</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Key Benefit</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">1. Training</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Features from feature tables are used to train the model; <code>fe.log_model</code> packages the model with feature metadata (which feature tables, features, and lookup keys it needs)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Full reproducibility &amp; traceability</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">2. Deployment</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Model is deployed to Model Serving; it knows which features it needs via the packaged metadata</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Zero manual feature wiring</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">3. Inference</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Model receives the primary key; feature lookup provides the rest (batch from the offline store, real-time from the online store), as long as the model was logged with <code>fe.log_model</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Simplified client — just send the key</td></tr></tbody></table></div>

### B5. Workspace FS vs. Unity Catalog FS

> ⚠️ **Databricks recommends using Feature Engineering in Unity Catalog. Workspace Feature Store is the legacy option; use it only to maintain existing workloads.**

---

<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 24px 0;">

<!-- Workspace FS -->
<div style="flex: 1; min-width: 320px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #5A6F77; background: #F0F2F4; border-bottom: 2px solid #5A6F77;">Workspace FS</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">
        <ul style="margin: 0; padding-left: 18px; line-height: 2;">
          <li>Specific to a <strong>single workspace</strong></li>
          <li>Challenges in sharing feature tables across workspaces</li>
          <li>Controls access using Databricks workspace access controls</li>
          <li>Uses <code>FeatureStoreClient</code> to create feature tables</li>
          <li><strong style="color: #FF5F46;">Has been deprecated</strong></li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>
</div>

<!-- Unity Catalog FS -->
<div style="flex: 1; min-width: 320px;">
<table style="border-collapse: separate; border-spacing: 0; width: 100%; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08);">
  <thead>
    <tr>
      <th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; color: #00A972; background: #E6F5EE; border-bottom: 2px solid #00A972;">Unity Catalog FS <span style="background: #00A972; color: #fff; padding: 2px 8px; border-radius: 4px; font-size: 10px; margin-left: 8px;">Recommended</span></th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #F9F7F4;">
      <td style="padding: 12px 16px; border-bottom: 1px solid #EEEDE9;">
        <ul style="margin: 0; padding-left: 18px; line-height: 2;">
          <li>Any Delta table with a primary key can be a feature table</li>
          <li>Enables <strong>sharing and access across workspaces</strong></li>
          <li>All UC capabilities available: discovery, governance, lineage</li>
          <li>More robust governance and larger-scale feature management</li>
          <li>Uses <code>FeatureEngineeringClient</code> to create feature tables</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>
</div>

</div>

---

<p style="font-size: 12px; color: #618794; margin-top: 8px;">Feature Engineering in Unity Catalog supports Unity Catalog features like discovery, governance, lineage, and access across workspaces. Any future updates to Unity Catalog's governance capabilities will automatically extend to feature data.</p>

### B6. Point-in-Time Feature Lookups

Many features change over time: a customer's average spend this month is different from last month. If you join each training label to the **latest** feature values, you leak information from the future into training. The model then looks accurate offline but fails in production, where future values do not exist yet.

A **point-in-time lookup** solves this. For each training row, it returns the feature values that were known **at or before** the row's timestamp.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 240" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Point-in-time feature lookup</title>
  <desc>A timeline for one customer shows three feature-table versions computed on January 1, February 1, and March 1. A training label observed on February 15 is joined to the February 1 feature values, the latest values known at that time, and not to the March 1 values, which would leak future information.</desc>
  <defs>
    <marker id="pit-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#00A972" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <text x="34" y="42" text-anchor="start" font-size="14" font-weight="700" fill="#FF5F46">Point-in-Time Lookup for customer_id = 42</text>
  <path d="M60 120 L800 120" fill="none" stroke="#90A5B1" stroke-width="2"/>
  <rect x="90" y="66" width="170" height="40" rx="8" fill="#ffffff" stroke="#618794" stroke-width="1.2"/>
  <text x="175" y="84" text-anchor="middle" font-size="12" font-weight="600" fill="#0b2026">Features as of Jan 1</text>
  <text x="175" y="99" text-anchor="middle" font-size="11" fill="#618794">avg_spend = 40</text>
  <circle cx="175" cy="120" r="5" fill="#618794"/>
  <rect x="320" y="66" width="170" height="40" rx="8" fill="#00A972" fill-opacity="0.14" stroke="#00A972" stroke-width="1.5"/>
  <text x="405" y="84" text-anchor="middle" font-size="12" font-weight="700" fill="#0b2026">Features as of Feb 1</text>
  <text x="405" y="99" text-anchor="middle" font-size="11" fill="#1B5162">avg_spend = 55</text>
  <circle cx="405" cy="120" r="6" fill="#00A972"/>
  <rect x="610" y="66" width="170" height="40" rx="8" fill="#ffffff" stroke="#98102A" stroke-width="1.2" stroke-dasharray="5 3"/>
  <text x="695" y="84" text-anchor="middle" font-size="12" font-weight="600" fill="#98102A">Features as of Mar 1</text>
  <text x="695" y="99" text-anchor="middle" font-size="11" fill="#98102A">future: must not be used</text>
  <circle cx="695" cy="120" r="5" fill="#98102A"/>
  <rect x="455" y="150" width="170" height="44" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="540" y="168" text-anchor="middle" font-size="12" font-weight="700" fill="#0b2026">Label observed Feb 15</text>
  <text x="540" y="184" text-anchor="middle" font-size="11" fill="#1B5162">churned = 1</text>
  <path d="M540 128 L540 148" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="3 3"/>
  <path d="M470 172 C 420 172, 405 160, 405 132" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#pit-arrow)"/>
  <text x="300" y="176" text-anchor="middle" font-size="11" fill="#00A972">joins the latest values</text>
  <text x="300" y="190" text-anchor="middle" font-size="11" fill="#00A972">at or before Feb 15</text>
</svg>
</div>

To enable point-in-time lookups, store the feature table as a **time series feature table** by declaring a timestamp column, then pass the matching timestamp column from your labels when you build the training set:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup</code>
</br>
<code>fe = FeatureEngineeringClient()</code>
</br>
<code># 1. Create a time series feature table (the timestamp column is part of the primary key)</code>
<code>fe.create_table(</code>
<code>    name="catalog.schema.customer_features",</code>
<code>    primary_keys=["customer_id", "feature_ts"],</code>
<code>    timeseries_columns="feature_ts",</code>
<code>    df=customer_features_df,</code>
<code>)</code>
</br>
<code># 2. Look up features as of each label's timestamp</code>
<code>training_set = fe.create_training_set(</code>
<code>    df=labels_df,  # customer_id, label_ts, churned</code>
<code>    feature_lookups=[</code>
<code>        FeatureLookup(</code>
<code>            table_name="catalog.schema.customer_features",</code>
<code>            lookup_key="customer_id",</code>
<code>            timestamp_lookup_key="label_ts",</code>
<code>        )</code>
<code>    ],</code>
<code>    label="churned",</code>
<code>)</code>
<code>training_df = training_set.load_df()</code></pre>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #0d47a1;">Info</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">The timestamp column must be a <code>TIMESTAMP</code> or <code>DATE</code> type. Point-in-time lookups are the feature-table version of the time-based split you saw in Module 2: both make sure the model is trained only on information that was available at prediction time.</div></div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;"><strong style="color: #0d47a1;">Learn More</strong><p style="margin: 8px 0 0 0; color: #333;">Point-in-time feature joins (<a href="https://docs.databricks.com/aws/en/machine-learning/feature-store/time-series" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/time-series" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/machine-learning/feature-store/time-series" style="color: #1976d2;">GCP</a>)</p></div>

### B7. Introduction to Lakebase: The Database Behind Online Features

So far, every feature table in this lecture has been a Delta table, which is ideal for training and batch scoring. Real-time models need something different: a database that can return one customer's features in milliseconds. On Databricks, that database is **Lakebase**. This section introduces it; the next section (B8) shows how you publish feature tables to it.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 160 1600 740" width="100%" style="max-width:1600px;height:auto;display:block;margin:0 auto">
  <defs>
    <style>
      @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&amp;display=swap');
      text { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
      .hb{fill:#1B3139;font-weight:800}.h{fill:#1B3139;font-weight:700}
      .body{fill:#1B3139;font-weight:400}.lbl{fill:#1B3139;font-weight:700}.meta{fill:#5A6A70;font-weight:500}
    </style>
    <filter id="tileShadow" x="-45%" y="-45%" width="190%" height="200%"><feDropShadow dx="0" dy="2.5" stdDeviation="4" flood-color="#1B3139" flood-opacity="0.13"/></filter>
    <filter id="softShadow" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="4" stdDeviation="10" flood-color="#1B3139" flood-opacity="0.10"/></filter>
    <symbol id="dbxMark" viewBox="0 0 105 113"><path d="M98.9 46.5996L52.3 72.8996L2.4 44.7996L0 46.0996V66.4996L52.3 95.8996L98.9 69.6996V80.4996L52.3 106.8L2.4 78.6996L0 79.9996V83.4996L52.3 112.9L104.5 83.4996V63.0996L102.1 61.7996L52.3 89.7996L5.6 63.5996V52.7996L52.3 78.9996L104.5 49.5996V29.4996L101.9 27.9996L52.3 55.8996L8 31.0996L52.3 6.19961L88.7 26.6996L91.9 24.8996V22.3996L52.3 0.0996094L0 29.4996V32.6996L52.3 62.0996L98.9 35.7996V46.5996Z" fill="#FF3621"/></symbol>
    <clipPath id="lakeClip"><rect x="580" y="402" width="446" height="164" rx="16"/></clipPath>
  </defs>
  <rect x="0" y="0" width="1600" height="900" fill="#FFFFFF"/>
  <rect x="196" y="196" width="1216" height="402" rx="18" fill="#FFFFFF" stroke="#C6DCEA" stroke-width="1.5" filter="url(#softShadow)"/>
  <text x="236" y="252" class="body" font-size="15">Separated storage</text>
  <text x="236" y="272" class="body" font-size="15">and compute, enabling</text>
  <text x="236" y="292" class="body" font-size="15">independent scale for</text>
  <text x="236" y="312" class="body" font-size="15">each resource</text>
  <text x="1156" y="258" class="body" font-size="15">Low-latency feature</text>
  <text x="1156" y="278" class="body" font-size="15">lookups, branching, and</text>
  <text x="1156" y="298" class="body" font-size="15">point-in-time restore</text>
  <rect x="458" y="222" width="206" height="124" rx="14" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="486" y="234" width="30" height="30" viewBox="192 59 512 512"><path d="M192 196.45C192 120.538 253.538 59.0001 329.45 59.0001H566.55C642.462 59.0001 704 120.538 704 196.45V433.55C704 509.462 642.462 571 566.55 571H329.45C253.538 571 192 509.462 192 433.55V196.45Z" fill="white"/><path d="M464.872 376.037H592.092V462.101H445.293L393.629 376.037H303.304V289.973L413.206 289.975L464.872 376.037Z" fill="#FF5F46"/><path d="M501.768 259.377L592.092 259.377V345.443L482.191 345.441L430.525 259.38L303.304 259.377V173.313L450.103 173.315L501.768 259.377Z" fill="#FABFBA"/></svg>
  <text x="524" y="256" class="hb" font-size="21">Lakeflow</text>
  <rect x="472.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="476.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="475.5" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="475.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="502.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="506.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="505.8" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="505.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="532.5" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="537.0" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="536.0" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="536.0" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="562.8" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="567.2" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="566.2" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="566.2" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="593.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="597.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="596.5" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="596.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="623.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="627.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="626.8" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="626.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="472.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="476.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="475.5" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="475.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="502.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="506.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="505.8" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="505.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="532.5" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="537.0" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="536.0" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="536.0" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="562.8" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="567.2" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="566.2" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="566.2" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="593.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="597.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="596.5" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="596.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="623.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="627.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="626.8" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="626.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="690" y="222" width="206" height="124" rx="14" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="718" y="234" width="30" height="30" viewBox="191 59 512 512"><path d="M191.359 196.33C191.359 120.418 252.898 58.88 328.809 58.88H565.91C641.821 58.88 703.359 120.418 703.359 196.33V433.43C703.359 509.342 641.821 570.88 565.91 570.88H328.809C252.898 570.88 191.359 509.342 191.359 433.43V196.33Z" fill="white"/><path d="M591.208 281.742V362.52H591.214L591.208 362.523V362.557H591.133C569.727 372.414 545.232 378.047 519.084 378.047C492.937 378.047 468.416 372.414 447.037 362.557H447.051C425.67 352.676 401.139 347.031 374.98 347.031C348.821 347.031 324.317 352.676 302.909 362.557H302.906V282.843H302.922V281.742L446.411 169.092L591.208 281.742Z" fill="#FABFBA"/><path d="M375.031 378.05C401.179 378.05 425.701 383.683 447.081 393.541H447.064C468.445 403.421 492.976 409.067 519.135 409.067C544.477 409.067 568.265 403.768 589.191 394.455L591.206 393.541V463.688H591.212C591.21 463.689 591.208 463.69 591.206 463.691V463.723H591.134C569.727 473.581 545.23 479.214 519.082 479.214C492.934 479.214 468.411 473.581 447.031 463.723H447.049C425.668 453.843 401.137 448.197 374.978 448.197C349.636 448.197 325.847 453.495 304.922 462.809L302.907 463.723H302.904L302.905 393.576V393.541H302.98C324.387 383.683 348.883 378.05 375.031 378.05Z" fill="#FF5F46"/></svg>
  <text x="756" y="256" class="hb" font-size="21">Lakehouse</text>
  <rect x="704.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="708.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="707.5" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="707.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="734.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="738.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="737.8" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="737.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="764.5" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="769.0" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="768.0" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="768.0" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="794.8" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="799.2" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="798.2" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="798.2" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="825.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="829.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="828.5" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="828.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="855.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="859.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="858.8" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="858.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="704.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="708.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="707.5" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="707.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="734.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="738.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="737.8" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="737.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="764.5" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="769.0" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="768.0" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="768.0" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="794.8" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="799.2" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="798.2" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="798.2" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="825.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="829.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="828.5" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="828.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="855.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="859.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="858.8" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="858.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="922" y="222" width="206" height="124" rx="14" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="950" y="234" width="30" height="30" viewBox="192 59 512 512"><path d="M192 196.45C192 120.538 253.538 59 329.45 59H566.55C642.462 59 704 120.538 704 196.45V433.55C704 509.462 642.462 571 566.55 571H329.45C253.538 571 192 509.462 192 433.55V196.45Z" fill="white"/><path d="M592.094 355.275H592.091C570.646 362.487 546.1 366.607 519.896 366.607C493.693 366.607 469.12 362.487 447.702 355.275H303.305V278.516H592.094V355.275ZM375.502 343.466C349.297 343.466 324.75 347.76 303.305 355.274H447.699C426.281 347.759 401.707 343.466 375.502 343.466Z" fill="#FABFBA"/><path d="M375.841 371.695C401.108 371.695 424.851 375.365 445.691 381.815L449.707 383.081C470.548 389.532 494.291 393.201 519.558 393.201C544.825 393.201 568.543 389.532 589.406 383.081L591.416 382.448H592.094V462.102H303.305V382.448H303.982C325.327 375.605 349.759 371.695 375.841 371.695Z" fill="#FF5F46"/><rect x="303.304" y="173.312" width="288.79" height="79.6536" fill="#FABFBA"/></svg>
  <text x="988" y="256" class="hb" font-size="21">Lakebase</text>
  <rect x="936.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="940.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="939.5" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="939.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="966.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="970.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="969.8" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="969.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="996.5" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1001.0" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="1000.0" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="1000.0" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1026.8" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1031.2" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="1030.2" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="1030.2" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1057.0" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1061.5" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="1060.5" y="286.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="1060.5" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1087.2" y="272.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1091.8" cy="276.6" r="1.2" fill="#FFFFFF"/><rect x="1090.8" y="286.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="1090.8" y="291.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="936.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="940.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="939.5" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="939.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="966.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="970.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="969.8" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="969.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="996.5" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1001.0" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="1000.0" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="1000.0" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1026.8" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1031.2" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="1030.2" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="1030.2" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1057.0" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1061.5" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="1060.5" y="318.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/><rect x="1060.5" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <rect x="1087.2" y="304.0" width="26.8" height="27" rx="3" fill="#F3D2CC"/><circle cx="1091.8" cy="308.6" r="1.2" fill="#FFFFFF"/><rect x="1090.8" y="318.0" width="19.8" height="2.4" rx="1.2" fill="#EE5540"/><rect x="1090.8" y="323.4" width="19.8" height="1.6" rx="0.8" fill="#FFFFFF"/>
  <svg x="260" y="360" width="27" height="27" viewBox="48 46 414 420"><path d="M181.185 46C204.241 46 223.38 63.8165 225.026 86.8145L238 268H126L137.316 87.208C138.766 64.0446 157.976 46.0001 181.185 46ZM405.266 114C428.468 114 447.914 131.542 450.297 154.622L462 268H350L360.179 155.202C362.284 131.871 381.84 114 405.266 114Z" fill="#FF5F46"/><path d="M338.996 424.79C337.367 448.002 318.062 466 294.793 466H294.792C271.695 465.999 252.471 448.258 250.623 425.234L238 268H350L338.996 424.79ZM128 268L120.132 380.564C118.973 397.143 105.185 410 88.5664 410C72.0694 410 58.3413 397.324 57.0264 380.88L48 268H128Z" fill="#FABFBA"/></svg>
  <text x="296" y="370" class="body" font-size="15">Real-time</text>
  <text x="296" y="388" class="body" font-size="15">streaming</text>
  <svg x="388" y="364" width="23" height="23" viewBox="64 64 385 385"><path d="M64 449.034L64.0015 266.742L146.336 266.742L146.336 365.752L244.527 365.752L244.527 449.034L64 449.034Z" fill="#FABFBA"/><path d="M447.992 64.0015L447.991 246.293L365.656 246.293L365.656 147.284L267.466 147.284L267.466 64.0015L447.992 64.0015Z" fill="#FABFBA"/><rect width="137.404" height="137.409" transform="matrix(1 0 5.39465e-06 1 64 64.001)" fill="#FF5F46"/><rect width="137.405" height="137.409" transform="matrix(1 0 5.39465e-06 1 311.744 311.665)" fill="#FF5F46"/></svg>
  <text x="418" y="382" class="body" font-size="15">Zerobus</text>
  <rect x="322.0" y="410.0" width="72.0" height="56.0" fill="#9DA3A6"/><ellipse cx="358.0" cy="466.0" rx="36.0" ry="10.8" fill="#9DA3A6"/><path d="M322.0,438.0 A36.0,10.8 0 0 0 394.0,438.0" fill="none" stroke="#FFFFFF" stroke-width="1.4" opacity="0.85"/><ellipse cx="358.0" cy="410.0" rx="36.0" ry="10.8" fill="#B7BBBD" stroke="#FFFFFF" stroke-width="1.2"/>
  <rect x="400.0" y="416.0" width="64.0" height="52.0" fill="#9DA3A6"/><ellipse cx="432.0" cy="468.0" rx="32.0" ry="9.6" fill="#9DA3A6"/><path d="M400.0,442.0 A32.0,9.6 0 0 0 464.0,442.0" fill="none" stroke="#FFFFFF" stroke-width="1.4" opacity="0.85"/><ellipse cx="432.0" cy="416.0" rx="32.0" ry="9.6" fill="#B7BBBD" stroke="#FFFFFF" stroke-width="1.2"/>
  <rect x="272.0" y="440.0" width="60.0" height="48.0" fill="#CFD2D3"/><ellipse cx="302.0" cy="488.0" rx="30.0" ry="9.0" fill="#CFD2D3"/><path d="M272.0,464.0 A30.0,9.0 0 0 0 332.0,464.0" fill="none" stroke="#FFFFFF" stroke-width="1.4" opacity="0.85"/><ellipse cx="302.0" cy="440.0" rx="30.0" ry="9.0" fill="#E7E8E9" stroke="#FFFFFF" stroke-width="1.2"/>
  <rect x="342.0" y="444.0" width="68.0" height="50.0" fill="#CFD2D3"/><ellipse cx="376.0" cy="494.0" rx="34.0" ry="10.2" fill="#CFD2D3"/><path d="M342.0,469.0 A34.0,10.2 0 0 0 410.0,469.0" fill="none" stroke="#FFFFFF" stroke-width="1.4" opacity="0.85"/><ellipse cx="376.0" cy="444.0" rx="34.0" ry="10.2" fill="#E7E8E9" stroke="#FFFFFF" stroke-width="1.2"/>
  <rect x="427.0" y="452.0" width="42.0" height="38.0" fill="#CFD2D3"/><ellipse cx="448.0" cy="490.0" rx="21.0" ry="6.3" fill="#CFD2D3"/><path d="M427.0,471.0 A21.0,6.3 0 0 0 469.0,471.0" fill="none" stroke="#FFFFFF" stroke-width="1.4" opacity="0.85"/><ellipse cx="448.0" cy="452.0" rx="21.0" ry="6.3" fill="#E7E8E9" stroke="#FFFFFF" stroke-width="1.2"/>
  <text x="376" y="516" text-anchor="middle" class="body" font-size="15">100+ supported</text>
  <text x="376" y="535" text-anchor="middle" class="body" font-size="15">data/app connectors</text>
  <path d="M470,432 L490,432 L490,352" fill="none" stroke="#52707E" stroke-width="2"/>
  <path d="M486.5,357 L490,350 L493.5,357" fill="#52707E"/>
  <line x1="636" y1="348" x2="636" y2="396" stroke="#52707E" stroke-width="2"/>  <path d="M632,396 L636,402 L640,396" fill="#52707E"/>
  <line x1="738" y1="354" x2="738" y2="396" stroke="#20A15B" stroke-width="2"/>  <path d="M734,354 L738,348 L742,354" fill="#20A15B"/>  <path d="M734,396 L738,402 L742,396" fill="#20A15B"/>
  <line x1="996" y1="354" x2="996" y2="396" stroke="#FF5F46" stroke-width="2"/>  <path d="M992,354 L996,348 L1000,354" fill="#FF5F46"/>  <path d="M992,396 L996,402 L1000,396" fill="#FF5F46"/>
  <text x="505" y="372" font-size="16" font-weight="700" fill="#52707E">Ingest, ETL</text>
  <text x="505" y="394" font-size="16" font-weight="700" fill="#52707E">&amp; streaming</text>
  <text x="758" y="372" font-size="16" font-weight="700" fill="#20A15B">Real-time +</text>
  <text x="758" y="394" font-size="16" font-weight="700" fill="#20A15B">batch analytics</text>
  <text x="1016" y="372" font-size="16" font-weight="700" fill="#FF5F46">Online feature</text>
  <text x="1016" y="394" font-size="16" font-weight="700" fill="#FF5F46">serving</text>
  <rect x="580" y="402" width="446" height="164" rx="16" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <rect x="600.0" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="609.0" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="668.8" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="677.8" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="737.7" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="746.7" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="806.5" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="815.5" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="875.3" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="884.3" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="944.2" y="421" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="953.2" cy="430.0" r="2.6" fill="#FFFFFF"/>
  <rect x="600.0" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="609.0" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="668.8" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="677.8" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="737.7" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="746.7" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="806.5" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="815.5" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="875.3" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="884.3" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="944.2" y="447" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="953.2" cy="456.0" r="2.6" fill="#FFFFFF"/>
  <rect x="600.0" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="609.0" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <rect x="668.8" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="677.8" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <rect x="737.7" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="746.7" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <rect x="806.5" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="815.5" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <rect x="875.3" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="884.3" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <rect x="944.2" y="473" width="61.8" height="18" rx="3" fill="#8FA3AD"/>
  <circle cx="953.2" cy="482.0" r="2.6" fill="#FFFFFF"/>
  <path d="M580,509.0 L583,507.6 L586,506.3 L589,505.1 L592,504.1 L595,503.3 L598,502.8 L601,502.5 L604,502.5 L607,502.9 L610,503.4 L613,504.3 L616,505.3 L619,506.5 L622,507.8 L625,509.2 L628,510.5 L631,511.8 L634,513.0 L637,514.0 L640,514.7 L643,515.3 L646,515.5 L649,515.4 L652,515.1 L655,514.5 L658,513.6 L661,512.5 L664,511.3 L667,510.0 L670,508.6 L673,507.3 L676,506.0 L679,504.9 L682,503.9 L685,503.2 L688,502.7 L691,502.5 L694,502.6 L697,503.0 L700,503.6 L703,504.5 L706,505.6 L709,506.8 L712,508.2 L715,509.5 L718,510.9 L721,512.2 L724,513.3 L727,514.2 L730,514.9 L733,515.3 L736,515.5 L739,515.4 L742,514.9 L745,514.3 L748,513.3 L751,512.2 L754,511.0 L757,509.6 L760,508.3 L763,506.9 L766,505.7 L769,504.6 L772,503.7 L775,503.0 L778,502.6 L781,502.5 L784,502.7 L787,503.1 L790,503.8 L793,504.8 L796,505.9 L799,507.2 L802,508.5 L805,509.9 L808,511.2 L811,512.5 L814,513.5 L817,514.4 L820,515.1 L823,515.4 L826,515.5 L829,515.3 L832,514.8 L835,514.0 L838,513.1 L841,511.9 L844,510.6 L847,509.3 L850,507.9 L853,506.6 L856,505.4 L859,504.3 L862,503.5 L865,502.9 L868,502.6 L871,502.5 L874,502.8 L877,503.3 L880,504.1 L883,505.1 L886,506.3 L889,507.5 L892,508.9 L895,510.3 L898,511.6 L901,512.8 L904,513.8 L907,514.6 L910,515.2 L913,515.5 L916,515.5 L919,515.2 L922,514.6 L925,513.8 L928,512.8 L931,511.6 L934,510.3 L937,508.9 L940,507.5 L943,506.3 L946,505.1 L949,504.1 L952,503.3 L955,502.8 L958,502.5 L961,502.6 L964,502.9 L967,503.5 L970,504.3 L973,505.4 L976,506.6 L979,507.9 L982,509.3 L985,510.6 L988,511.9 L991,513.1 L994,514.0 L997,514.8 L1000,515.3 L1003,515.5 L1006,515.4 L1009,515.1 L1012,514.4 L1015,513.5 L1018,512.5 L1021,511.2 L1024,509.9 L1026,566 L580,566 Z" fill="#16495B" clip-path="url(#lakeClip)"/>
  <svg x="712" y="534" width="77" height="16" viewBox="6.5 31.4 176.3 36.4"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#FFFFFF"/><path d="M49.1079 56.876V42.2803C49.1079 42.1509 49.1593 42.0215 49.2621 41.918C49.3648 41.8145 49.4933 41.7627 49.6218 41.7627H55.4028C57.6124 41.7627 59.4366 42.4097 60.8497 43.7295C61.6205 44.4282 62.1858 45.2822 62.5969 46.2656C62.9823 47.249 63.1878 48.3618 63.1878 49.5781C63.1878 50.7944 62.9823 51.9072 62.5969 52.8906C62.2115 53.874 61.6205 54.728 60.8497 55.4268C60.156 56.0737 59.3338 56.5395 58.4346 56.876C57.5353 57.2124 56.5076 57.3677 55.4284 57.3677H49.6475C49.519 57.3677 49.3905 57.3159 49.2878 57.2124C49.1593 57.1348 49.1079 57.0054 49.1079 56.876ZM53.8098 53.46H55.4028C55.7882 53.46 56.1992 53.4082 56.636 53.3047C57.0471 53.2012 57.4325 53.02 57.7665 52.7354C58.1005 52.4507 58.3832 52.0625 58.6144 51.5708C58.82 51.0532 58.9484 50.4063 58.9484 49.604C58.9484 48.8018 58.8456 48.1289 58.6144 47.6372C58.3832 47.1196 58.1005 46.7314 57.7665 46.4727C57.4325 46.188 57.0471 46.0068 56.636 45.9033C56.2249 45.7998 55.8138 45.748 55.4284 45.748H53.8098C53.6813 45.748 53.5528 45.7998 53.4501 45.9033C53.3473 46.0068 53.2959 46.1362 53.2959 46.2656V52.9683C53.2959 53.1235 53.3473 53.227 53.4501 53.3306C53.5271 53.4082 53.6556 53.46 53.8098 53.46Z" fill="#FFFFFF"/><path d="M69.2256 46.2139V47.1196C69.2256 47.249 69.277 47.3784 69.3797 47.4819C69.4825 47.5854 69.611 47.6372 69.7394 47.6372H75.469C75.5975 47.6372 75.726 47.689 75.8287 47.7925C75.9315 47.896 75.9829 48.0254 75.9829 48.1548V51.105C75.9829 51.2344 75.9315 51.3638 75.8287 51.4673C75.726 51.5708 75.5975 51.6226 75.469 51.6226H69.7394C69.611 51.6226 69.4825 51.6743 69.3797 51.7778C69.277 51.8813 69.2256 52.0107 69.2256 52.1401V52.9424C69.2256 53.0977 69.277 53.2012 69.3797 53.3047C69.4825 53.4082 69.611 53.46 69.7394 53.46H76.3169C76.4454 53.46 76.5738 53.5117 76.6766 53.6152C76.7794 53.7187 76.8308 53.8481 76.8308 53.9775V56.9277C76.8308 57.083 76.7794 57.1865 76.6766 57.29C76.5738 57.3936 76.4454 57.4453 76.3169 57.4453H65.5515C65.423 57.4453 65.2945 57.3936 65.1918 57.29C65.089 57.1865 65.0376 57.0571 65.0376 56.9277V42.2803C65.0376 42.1509 65.089 42.0215 65.1918 41.918C65.2945 41.8145 65.423 41.7627 65.5515 41.7627H76.3169C76.4454 41.7627 76.5738 41.8145 76.6766 41.918C76.7794 42.0215 76.8308 42.1509 76.8308 42.2803V45.1787C76.8308 45.334 76.7794 45.4375 76.6766 45.541C76.5738 45.6445 76.4454 45.6963 76.3169 45.6963H69.7394C69.611 45.6963 69.4825 45.748 69.3797 45.8516C69.2513 45.9551 69.2256 46.0586 69.2256 46.2139Z" fill="#FFFFFF"/><path d="M91.0649 53.9775V56.876C91.0649 57.0312 91.0135 57.1348 90.9107 57.2383C90.8079 57.3418 90.6795 57.3936 90.551 57.3936H79.9911C79.8369 57.3936 79.7085 57.3418 79.6314 57.2383C79.5286 57.1348 79.5029 57.0054 79.5029 56.876V42.2803C79.5029 42.1509 79.5543 42.0215 79.6314 41.918C79.7342 41.8145 79.8369 41.7627 79.9911 41.7627H83.1771C83.3055 41.7627 83.434 41.8145 83.5368 41.918C83.6395 42.0215 83.6909 42.1509 83.6909 42.2803V52.9424C83.6909 53.0977 83.7423 53.2012 83.8194 53.3047C83.9222 53.4082 84.0249 53.46 84.1791 53.46H90.551C90.6795 53.46 90.8079 53.5117 90.9107 53.6152C91.0135 53.7187 91.0649 53.8223 91.0649 53.9775Z" fill="#FFFFFF"/><path d="M100.905 45.6963H97.9507C97.8222 45.6963 97.6938 45.748 97.591 45.8516C97.4882 45.9551 97.4369 46.0845 97.4369 46.2139V56.876C97.4369 57.0312 97.3855 57.1348 97.3084 57.2383C97.2056 57.3418 97.1028 57.3936 96.9487 57.3936H93.7627C93.6343 57.3936 93.5058 57.3418 93.403 57.2383C93.3003 57.1348 93.2489 57.0054 93.2489 56.876V46.2139C93.2489 46.0845 93.1975 45.9551 93.1204 45.8516C93.0176 45.748 92.9149 45.6963 92.7607 45.6963H89.7546C89.6004 45.6963 89.4977 45.6445 89.3949 45.541C89.2921 45.4375 89.2407 45.3081 89.2407 45.1787V42.2803C89.2407 42.1509 89.2921 42.0215 89.3949 41.918C89.4977 41.8145 89.6004 41.7627 89.7546 41.7627H100.931C101.085 41.7627 101.214 41.8145 101.291 41.918C101.394 42.0215 101.419 42.1509 101.419 42.2803V45.1787C101.419 45.334 101.368 45.4375 101.291 45.541C101.188 45.6445 101.06 45.6963 100.905 45.6963Z" fill="#FFFFFF"/><path d="M105.967 42.2539C106.018 42.1245 106.121 42.021 106.25 41.9175C106.378 41.814 106.532 41.7881 106.687 41.7881H109.641C109.77 41.7881 109.924 41.8398 110.052 41.9175C110.181 42.021 110.284 42.1245 110.335 42.2539L116.527 56.7461C116.578 56.8755 116.578 57.0049 116.501 57.1602C116.424 57.2896 116.27 57.3672 116.039 57.3672H112.416C112.262 57.3672 112.108 57.3154 111.979 57.2378C111.851 57.1343 111.748 57.0308 111.671 56.9014L111.106 55.6333C111.029 55.5039 110.926 55.4004 110.797 55.2969C110.669 55.1934 110.515 55.1675 110.386 55.1675H105.916C105.762 55.1675 105.607 55.2192 105.479 55.2969C105.35 55.4004 105.248 55.5039 105.196 55.6333L104.605 56.9014C104.554 57.0308 104.451 57.1343 104.323 57.2378C104.194 57.3413 104.04 57.3672 103.912 57.3672H100.263C100.186 57.3672 100.109 57.3413 100.032 57.3154C99.9549 57.2896 99.9035 57.2378 99.8521 57.186C99.8007 57.1343 99.775 57.0566 99.7493 56.979C99.7236 56.9014 99.7493 56.7979 99.8007 56.6685L105.967 42.2539ZM107.226 51.5444H109.102C109.23 51.5444 109.333 51.4927 109.41 51.415C109.461 51.3115 109.461 51.208 109.41 51.0786C109.282 50.7163 109.102 50.3022 108.896 49.8364C108.691 49.3706 108.511 48.9048 108.357 48.4907C108.305 48.3613 108.228 48.3096 108.177 48.3096C108.1 48.3096 108.048 48.3613 107.997 48.4907L106.943 51.0786C106.892 51.208 106.892 51.3115 106.943 51.415C106.969 51.4927 107.072 51.5444 107.226 51.5444Z" fill="#FFFFFF"/><path d="M136.825 53.9775V56.876C136.825 57.0312 136.773 57.1348 136.671 57.2383C136.568 57.3418 136.439 57.3936 136.311 57.3936H125.751C125.597 57.3936 125.468 57.3418 125.391 57.2383C125.288 57.1348 125.263 57.0054 125.263 56.876V42.2803C125.263 42.1509 125.314 42.0215 125.391 41.918C125.494 41.8145 125.597 41.7627 125.751 41.7627H128.937C129.065 41.7627 129.194 41.8145 129.297 41.918C129.399 42.0215 129.451 42.1509 129.451 42.2803V52.9424C129.451 53.0977 129.502 53.2012 129.579 53.3047C129.682 53.4082 129.785 53.46 129.939 53.46H136.311C136.439 53.46 136.568 53.5117 136.671 53.6152C136.773 53.7187 136.825 53.8223 136.825 53.9775Z" fill="#FFFFFF"/><path d="M143.351 42.2539C143.402 42.1245 143.505 42.021 143.633 41.9175C143.762 41.814 143.916 41.7881 144.07 41.7881H147.025C147.153 41.7881 147.308 41.8398 147.436 41.9175C147.565 42.021 147.667 42.1245 147.719 42.2539L153.911 56.7461C153.962 56.8755 153.962 57.0049 153.885 57.1602C153.808 57.2896 153.654 57.3672 153.423 57.3672H149.8C149.646 57.3672 149.492 57.3154 149.363 57.2378C149.235 57.1343 149.132 57.0308 149.055 56.9014L148.49 55.6333C148.412 55.5039 148.31 55.4004 148.181 55.2969C148.053 55.1934 147.899 55.1675 147.77 55.1675H143.299C143.145 55.1675 142.991 55.2192 142.863 55.2969C142.734 55.4004 142.631 55.5039 142.58 55.6333L141.989 56.9014C141.938 57.0308 141.835 57.1343 141.707 57.2378C141.578 57.3413 141.424 57.3672 141.295 57.3672H137.647C137.57 57.3672 137.493 57.3413 137.416 57.3154C137.339 57.2896 137.287 57.2378 137.236 57.186C137.185 57.1343 137.159 57.0566 137.133 56.979C137.107 56.9014 137.133 56.7979 137.185 56.6685L143.351 42.2539ZM144.584 51.5444H146.46C146.588 51.5444 146.691 51.4927 146.768 51.415C146.819 51.3115 146.819 51.208 146.768 51.0786C146.64 50.7163 146.46 50.3022 146.254 49.8364C146.049 49.3706 145.869 48.9048 145.715 48.4907C145.663 48.3613 145.586 48.3096 145.535 48.3096C145.483 48.3096 145.406 48.3613 145.355 48.4907L144.302 51.0786C144.25 51.208 144.25 51.3115 144.302 51.415C144.327 51.4927 144.43 51.5444 144.584 51.5444Z" fill="#FFFFFF"/><path d="M164.085 57.0054L159.974 52.373C159.871 52.2695 159.794 52.2178 159.743 52.2437C159.692 52.2695 159.64 52.3472 159.64 52.5024V56.876C159.64 57.0312 159.589 57.1348 159.486 57.2383C159.383 57.3418 159.255 57.3936 159.126 57.3936H155.94C155.786 57.3936 155.658 57.3418 155.581 57.2383C155.478 57.1348 155.452 57.0054 155.452 56.876V42.2803C155.452 42.1509 155.504 42.0215 155.581 41.918C155.683 41.8145 155.786 41.7627 155.94 41.7627H159.126C159.255 41.7627 159.383 41.8145 159.486 41.918C159.589 42.0215 159.64 42.1509 159.64 42.2803V47.1973C159.64 47.3525 159.666 47.4302 159.717 47.4302C159.769 47.4561 159.846 47.4043 159.923 47.3008L163.725 42.2026C163.802 42.0991 163.931 41.9956 164.085 41.918C164.239 41.8403 164.393 41.7886 164.548 41.7886H168.581C168.736 41.7886 168.838 41.8662 168.89 41.9956C168.941 42.125 168.915 42.2544 168.838 42.3579L163.289 49.397C163.211 49.5005 163.16 49.6299 163.16 49.7852C163.16 49.9404 163.212 50.0698 163.314 50.1475C163.802 50.665 164.291 51.2085 164.83 51.7778C165.37 52.3472 165.909 52.9165 166.423 53.4858C166.963 54.0552 167.477 54.6245 167.99 55.1938C168.504 55.7632 168.992 56.2808 169.455 56.7725C169.558 56.876 169.583 57.0054 169.558 57.1606C169.532 57.29 169.429 57.3677 169.275 57.3677H164.882C164.727 57.3677 164.573 57.3418 164.393 57.2642C164.291 57.2124 164.188 57.1089 164.085 57.0054Z" fill="#FFFFFF"/><path d="M175.133 46.2139V47.1196C175.133 47.249 175.184 47.3784 175.287 47.4819C175.39 47.5854 175.493 47.6372 175.647 47.6372H181.376C181.505 47.6372 181.633 47.689 181.736 47.7925C181.839 47.896 181.89 48.0254 181.89 48.1548V51.105C181.89 51.2344 181.839 51.3638 181.736 51.4673C181.633 51.5708 181.505 51.6226 181.376 51.6226H175.647C175.493 51.6226 175.39 51.6743 175.287 51.7778C175.184 51.8813 175.133 52.0107 175.133 52.1401V52.9424C175.133 53.0977 175.184 53.2012 175.287 53.3047C175.39 53.4082 175.493 53.46 175.647 53.46H182.224C182.353 53.46 182.481 53.5117 182.584 53.6152C182.687 53.7187 182.738 53.8481 182.738 53.9775V56.9277C182.738 57.083 182.687 57.1865 182.584 57.29C182.481 57.3936 182.353 57.4453 182.224 57.4453H171.459C171.33 57.4453 171.202 57.3936 171.099 57.29C170.996 57.1865 170.945 57.0571 170.945 56.9277V42.2803C170.945 42.1509 170.996 42.0215 171.099 41.918C171.202 41.8145 171.33 41.7627 171.459 41.7627H182.224C182.353 41.7627 182.481 41.8145 182.584 41.918C182.687 42.0215 182.738 42.1509 182.738 42.2803V45.1787C182.738 45.334 182.687 45.4375 182.584 45.541C182.481 45.6445 182.353 45.6963 182.224 45.6963H175.647C175.493 45.6963 175.39 45.748 175.287 45.8516C175.184 45.9551 175.133 46.0586 175.133 46.2139Z" fill="#FFFFFF"/></svg>
  <svg x="842" y="535" width="49" height="15" viewBox="0 0 168 51.3"><path d="M5.53331 39.1625H0V12.6328H5.53331V39.1625Z" fill="#FFFFFF"/><path d="M10.2703 25.8961C10.2703 17.7871 16.1914 12.1384 24.1021 12.1384C29.1054 12.1384 33.1585 14.3736 35.4322 17.4674L30.6538 20.5908C29.2505 18.6575 26.9028 17.4467 24.0961 17.4467C19.3592 17.4467 16.0227 20.8957 16.0227 25.8991C16.0227 30.9025 19.3207 34.3515 24.0577 34.3515C26.9383 34.3515 29.4755 33.0637 30.9173 30.6745L35.5417 33.9311C33.117 37.5697 28.8331 39.6539 24.0577 39.6539C16.147 39.6539 10.2703 34.0081 10.2703 25.8961Z" fill="#FFFFFF"/><path d="M60.1411 34.0437V39.1595H40.2816V12.6328H59.8362V17.7486H45.8149V23.1665H59.2293V28.3238H45.8149V34.0466L60.1411 34.0437Z" fill="#FFFFFF"/><path d="M87.2007 31.6574C87.2007 35.9384 84.0181 39.1625 78.8637 39.1625H64.875V12.6328H77.9874C82.9907 12.6328 86.0224 15.5519 86.0224 19.7559C86.0224 22.5211 84.6191 24.6053 82.1559 25.4787C85.3414 26.0471 87.2007 28.3238 87.2007 31.6574ZM70.4083 17.3312V23.3589H77.0015C79.1627 23.3589 80.2581 22.1747 80.2581 20.1763C80.2581 18.4 79.1598 17.3342 77.0015 17.3342L70.4083 17.3312ZM81.4364 31.163C81.4364 29.1173 80.2522 27.9804 77.9489 27.9804H70.4083V34.4611H77.9489C80.2611 34.4611 81.4364 33.2502 81.4364 31.166V31.163Z" fill="#FFFFFF"/><path d="M111.791 34.0437V39.1595H91.9376V12.6328H111.477V17.7486H97.465V23.1665H110.885V28.3238H97.465V34.0466L111.791 34.0437Z" fill="#FFFFFF"/><path d="M125.623 29.4962H122.07V39.1595H116.528V12.6328H128.391C134.114 12.6328 137.602 16.0818 137.602 21.0467C137.602 25.0257 135.366 28.0574 131.5 29.1173L138.398 39.1595H131.766L125.623 29.4962ZM122.07 24.3418H128.249C130.635 24.3418 131.888 23.205 131.888 21.0852C131.888 18.9654 130.638 17.7871 128.249 17.7871H122.059L122.07 24.3418Z" fill="#FFFFFF"/><path d="M154.613 39.6539C146.694 39.6539 140.781 34.0081 140.781 25.8606C140.781 17.7131 146.806 12.1413 154.729 12.1413C158.873 12.1413 162.648 13.6571 165.387 16.7272L161.094 20.3273C160.312 19.3995 159.331 18.659 158.225 18.1605C157.119 17.6619 155.915 17.418 154.702 17.4467C149.814 17.4467 146.554 20.9342 146.554 25.8991C146.554 30.864 149.775 34.3515 154.74 34.3515C158.379 34.3515 160.958 32.4183 161.828 28.9692H154.702V24.3803H167.853C167.949 25.02 167.987 25.6671 167.965 26.3136C168.024 34.1976 162.42 39.6539 154.613 39.6539Z" fill="#FFFFFF"/></svg>
  <text x="1044" y="432" class="body" font-size="15">Unified, open-format data lake</text>
  <text x="1044" y="452" class="body" font-size="15">for ALL business data</text>
  <text x="1044" y="484" class="body" font-size="15">Offline feature tables</text>
  <text x="1044" y="504" class="body" font-size="15">live here as Delta tables</text>
  <path d="M804,598 L804,652" fill="none" stroke="#A9C4D6" stroke-width="1.5"/>
  <rect x="96" y="652" width="1408" height="164" rx="20" fill="#FFFFFF" filter="url(#softShadow)"/>
  <rect x="110" y="664" width="1380" height="76" rx="14" fill="#EBF1F6" stroke="#C6DCEA" stroke-width="1.2"/>
  <text x="132" y="710" class="hb" font-size="23">Data for ML</text>
  <rect x="436" y="678" width="48" height="48" rx="13" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="436" y="678" width="48" height="48" viewBox="192 59 512 512"><path d="M192 196.45C192 120.538 253.538 59.0001 329.45 59.0001H566.55C642.462 59.0001 704 120.538 704 196.45V433.55C704 509.462 642.462 571 566.55 571H329.45C253.538 571 192 509.462 192 433.55V196.45Z" fill="white"/><path d="M464.872 376.037H592.092V462.101H445.293L393.629 376.037H303.304V289.973L413.206 289.975L464.872 376.037Z" fill="#FF5F46"/><path d="M501.768 259.377L592.092 259.377V345.443L482.191 345.441L430.525 259.38L303.304 259.377V173.313L450.103 173.315L501.768 259.377Z" fill="#FABFBA"/></svg>
  <text x="500" y="710" class="hb" font-size="23">Lakeflow</text>
  <rect x="772" y="678" width="48" height="48" rx="13" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="772" y="678" width="48" height="48" viewBox="191 59 512 512"><path d="M191.359 196.33C191.359 120.418 252.898 58.88 328.809 58.88H565.91C641.821 58.88 703.359 120.418 703.359 196.33V433.43C703.359 509.342 641.821 570.88 565.91 570.88H328.809C252.898 570.88 191.359 509.342 191.359 433.43V196.33Z" fill="white"/><path d="M591.208 281.742V362.52H591.214L591.208 362.523V362.557H591.133C569.727 372.414 545.232 378.047 519.084 378.047C492.937 378.047 468.416 372.414 447.037 362.557H447.051C425.67 352.676 401.139 347.031 374.98 347.031C348.821 347.031 324.317 352.676 302.909 362.557H302.906V282.843H302.922V281.742L446.411 169.092L591.208 281.742Z" fill="#FABFBA"/><path d="M375.031 378.05C401.179 378.05 425.701 383.683 447.081 393.541H447.064C468.445 403.421 492.976 409.067 519.135 409.067C544.477 409.067 568.265 403.768 589.191 394.455L591.206 393.541V463.688H591.212C591.21 463.689 591.208 463.69 591.206 463.691V463.723H591.134C569.727 473.581 545.23 479.214 519.082 479.214C492.934 479.214 468.411 473.581 447.031 463.723H447.049C425.668 453.843 401.137 448.197 374.978 448.197C349.636 448.197 325.847 453.495 304.922 462.809L302.907 463.723H302.904L302.905 393.576V393.541H302.98C324.387 383.683 348.883 378.05 375.031 378.05Z" fill="#FF5F46"/></svg>
  <text x="836" y="710" class="hb" font-size="23">Lakehouse</text>
  <rect x="1088" y="678" width="48" height="48" rx="13" fill="#FFFFFF" filter="url(#tileShadow)"/>
  <svg x="1088" y="678" width="48" height="48" viewBox="192 59 512 512"><path d="M192 196.45C192 120.538 253.538 59 329.45 59H566.55C642.462 59 704 120.538 704 196.45V433.55C704 509.462 642.462 571 566.55 571H329.45C253.538 571 192 509.462 192 433.55V196.45Z" fill="white"/><path d="M592.094 355.275H592.091C570.646 362.487 546.1 366.607 519.896 366.607C493.693 366.607 469.12 362.487 447.702 355.275H303.305V278.516H592.094V355.275ZM375.502 343.466C349.297 343.466 324.75 347.76 303.305 355.274H447.699C426.281 347.759 401.707 343.466 375.502 343.466Z" fill="#FABFBA"/><path d="M375.841 371.695C401.108 371.695 424.851 375.365 445.691 381.815L449.707 383.081C470.548 389.532 494.291 393.201 519.558 393.201C544.825 393.201 568.543 389.532 589.406 383.081L591.416 382.448H592.094V462.102H303.305V382.448H303.982C325.327 375.605 349.759 371.695 375.841 371.695Z" fill="#FF5F46"/><rect x="303.304" y="173.312" width="288.79" height="79.6536" fill="#FABFBA"/></svg>
  <text x="1152" y="710" class="hb" font-size="23">Lakebase</text>
  <rect x="110" y="752" width="1380" height="50" rx="12" fill="#EBF1F6" stroke="#C6DCEA" stroke-width="1.2"/>
  <text x="132" y="782" class="hb" font-size="19">Open Infrastructure</text>
  <text x="486" y="781" class="lbl" font-size="14">Open Format Data Lake</text>
  <svg x="676" y="761" width="97" height="20" viewBox="6.5 31.4 176.3 36.4"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/><path d="M49.1079 56.876V42.2803C49.1079 42.1509 49.1593 42.0215 49.2621 41.918C49.3648 41.8145 49.4933 41.7627 49.6218 41.7627H55.4028C57.6124 41.7627 59.4366 42.4097 60.8497 43.7295C61.6205 44.4282 62.1858 45.2822 62.5969 46.2656C62.9823 47.249 63.1878 48.3618 63.1878 49.5781C63.1878 50.7944 62.9823 51.9072 62.5969 52.8906C62.2115 53.874 61.6205 54.728 60.8497 55.4268C60.156 56.0737 59.3338 56.5395 58.4346 56.876C57.5353 57.2124 56.5076 57.3677 55.4284 57.3677H49.6475C49.519 57.3677 49.3905 57.3159 49.2878 57.2124C49.1593 57.1348 49.1079 57.0054 49.1079 56.876ZM53.8098 53.46H55.4028C55.7882 53.46 56.1992 53.4082 56.636 53.3047C57.0471 53.2012 57.4325 53.02 57.7665 52.7354C58.1005 52.4507 58.3832 52.0625 58.6144 51.5708C58.82 51.0532 58.9484 50.4063 58.9484 49.604C58.9484 48.8018 58.8456 48.1289 58.6144 47.6372C58.3832 47.1196 58.1005 46.7314 57.7665 46.4727C57.4325 46.188 57.0471 46.0068 56.636 45.9033C56.2249 45.7998 55.8138 45.748 55.4284 45.748H53.8098C53.6813 45.748 53.5528 45.7998 53.4501 45.9033C53.3473 46.0068 53.2959 46.1362 53.2959 46.2656V52.9683C53.2959 53.1235 53.3473 53.227 53.4501 53.3306C53.5271 53.4082 53.6556 53.46 53.8098 53.46Z" fill="#302D2D"/><path d="M69.2256 46.2139V47.1196C69.2256 47.249 69.277 47.3784 69.3797 47.4819C69.4825 47.5854 69.611 47.6372 69.7394 47.6372H75.469C75.5975 47.6372 75.726 47.689 75.8287 47.7925C75.9315 47.896 75.9829 48.0254 75.9829 48.1548V51.105C75.9829 51.2344 75.9315 51.3638 75.8287 51.4673C75.726 51.5708 75.5975 51.6226 75.469 51.6226H69.7394C69.611 51.6226 69.4825 51.6743 69.3797 51.7778C69.277 51.8813 69.2256 52.0107 69.2256 52.1401V52.9424C69.2256 53.0977 69.277 53.2012 69.3797 53.3047C69.4825 53.4082 69.611 53.46 69.7394 53.46H76.3169C76.4454 53.46 76.5738 53.5117 76.6766 53.6152C76.7794 53.7187 76.8308 53.8481 76.8308 53.9775V56.9277C76.8308 57.083 76.7794 57.1865 76.6766 57.29C76.5738 57.3936 76.4454 57.4453 76.3169 57.4453H65.5515C65.423 57.4453 65.2945 57.3936 65.1918 57.29C65.089 57.1865 65.0376 57.0571 65.0376 56.9277V42.2803C65.0376 42.1509 65.089 42.0215 65.1918 41.918C65.2945 41.8145 65.423 41.7627 65.5515 41.7627H76.3169C76.4454 41.7627 76.5738 41.8145 76.6766 41.918C76.7794 42.0215 76.8308 42.1509 76.8308 42.2803V45.1787C76.8308 45.334 76.7794 45.4375 76.6766 45.541C76.5738 45.6445 76.4454 45.6963 76.3169 45.6963H69.7394C69.611 45.6963 69.4825 45.748 69.3797 45.8516C69.2513 45.9551 69.2256 46.0586 69.2256 46.2139Z" fill="#302D2D"/><path d="M91.0649 53.9775V56.876C91.0649 57.0312 91.0135 57.1348 90.9107 57.2383C90.8079 57.3418 90.6795 57.3936 90.551 57.3936H79.9911C79.8369 57.3936 79.7085 57.3418 79.6314 57.2383C79.5286 57.1348 79.5029 57.0054 79.5029 56.876V42.2803C79.5029 42.1509 79.5543 42.0215 79.6314 41.918C79.7342 41.8145 79.8369 41.7627 79.9911 41.7627H83.1771C83.3055 41.7627 83.434 41.8145 83.5368 41.918C83.6395 42.0215 83.6909 42.1509 83.6909 42.2803V52.9424C83.6909 53.0977 83.7423 53.2012 83.8194 53.3047C83.9222 53.4082 84.0249 53.46 84.1791 53.46H90.551C90.6795 53.46 90.8079 53.5117 90.9107 53.6152C91.0135 53.7187 91.0649 53.8223 91.0649 53.9775Z" fill="#302D2D"/><path d="M100.905 45.6963H97.9507C97.8222 45.6963 97.6938 45.748 97.591 45.8516C97.4882 45.9551 97.4369 46.0845 97.4369 46.2139V56.876C97.4369 57.0312 97.3855 57.1348 97.3084 57.2383C97.2056 57.3418 97.1028 57.3936 96.9487 57.3936H93.7627C93.6343 57.3936 93.5058 57.3418 93.403 57.2383C93.3003 57.1348 93.2489 57.0054 93.2489 56.876V46.2139C93.2489 46.0845 93.1975 45.9551 93.1204 45.8516C93.0176 45.748 92.9149 45.6963 92.7607 45.6963H89.7546C89.6004 45.6963 89.4977 45.6445 89.3949 45.541C89.2921 45.4375 89.2407 45.3081 89.2407 45.1787V42.2803C89.2407 42.1509 89.2921 42.0215 89.3949 41.918C89.4977 41.8145 89.6004 41.7627 89.7546 41.7627H100.931C101.085 41.7627 101.214 41.8145 101.291 41.918C101.394 42.0215 101.419 42.1509 101.419 42.2803V45.1787C101.419 45.334 101.368 45.4375 101.291 45.541C101.188 45.6445 101.06 45.6963 100.905 45.6963Z" fill="#302D2D"/><path d="M105.967 42.2539C106.018 42.1245 106.121 42.021 106.25 41.9175C106.378 41.814 106.532 41.7881 106.687 41.7881H109.641C109.77 41.7881 109.924 41.8398 110.052 41.9175C110.181 42.021 110.284 42.1245 110.335 42.2539L116.527 56.7461C116.578 56.8755 116.578 57.0049 116.501 57.1602C116.424 57.2896 116.27 57.3672 116.039 57.3672H112.416C112.262 57.3672 112.108 57.3154 111.979 57.2378C111.851 57.1343 111.748 57.0308 111.671 56.9014L111.106 55.6333C111.029 55.5039 110.926 55.4004 110.797 55.2969C110.669 55.1934 110.515 55.1675 110.386 55.1675H105.916C105.762 55.1675 105.607 55.2192 105.479 55.2969C105.35 55.4004 105.248 55.5039 105.196 55.6333L104.605 56.9014C104.554 57.0308 104.451 57.1343 104.323 57.2378C104.194 57.3413 104.04 57.3672 103.912 57.3672H100.263C100.186 57.3672 100.109 57.3413 100.032 57.3154C99.9549 57.2896 99.9035 57.2378 99.8521 57.186C99.8007 57.1343 99.775 57.0566 99.7493 56.979C99.7236 56.9014 99.7493 56.7979 99.8007 56.6685L105.967 42.2539ZM107.226 51.5444H109.102C109.23 51.5444 109.333 51.4927 109.41 51.415C109.461 51.3115 109.461 51.208 109.41 51.0786C109.282 50.7163 109.102 50.3022 108.896 49.8364C108.691 49.3706 108.511 48.9048 108.357 48.4907C108.305 48.3613 108.228 48.3096 108.177 48.3096C108.1 48.3096 108.048 48.3613 107.997 48.4907L106.943 51.0786C106.892 51.208 106.892 51.3115 106.943 51.415C106.969 51.4927 107.072 51.5444 107.226 51.5444Z" fill="#302D2D"/><path d="M136.825 53.9775V56.876C136.825 57.0312 136.773 57.1348 136.671 57.2383C136.568 57.3418 136.439 57.3936 136.311 57.3936H125.751C125.597 57.3936 125.468 57.3418 125.391 57.2383C125.288 57.1348 125.263 57.0054 125.263 56.876V42.2803C125.263 42.1509 125.314 42.0215 125.391 41.918C125.494 41.8145 125.597 41.7627 125.751 41.7627H128.937C129.065 41.7627 129.194 41.8145 129.297 41.918C129.399 42.0215 129.451 42.1509 129.451 42.2803V52.9424C129.451 53.0977 129.502 53.2012 129.579 53.3047C129.682 53.4082 129.785 53.46 129.939 53.46H136.311C136.439 53.46 136.568 53.5117 136.671 53.6152C136.773 53.7187 136.825 53.8223 136.825 53.9775Z" fill="#302D2D"/><path d="M143.351 42.2539C143.402 42.1245 143.505 42.021 143.633 41.9175C143.762 41.814 143.916 41.7881 144.07 41.7881H147.025C147.153 41.7881 147.308 41.8398 147.436 41.9175C147.565 42.021 147.667 42.1245 147.719 42.2539L153.911 56.7461C153.962 56.8755 153.962 57.0049 153.885 57.1602C153.808 57.2896 153.654 57.3672 153.423 57.3672H149.8C149.646 57.3672 149.492 57.3154 149.363 57.2378C149.235 57.1343 149.132 57.0308 149.055 56.9014L148.49 55.6333C148.412 55.5039 148.31 55.4004 148.181 55.2969C148.053 55.1934 147.899 55.1675 147.77 55.1675H143.299C143.145 55.1675 142.991 55.2192 142.863 55.2969C142.734 55.4004 142.631 55.5039 142.58 55.6333L141.989 56.9014C141.938 57.0308 141.835 57.1343 141.707 57.2378C141.578 57.3413 141.424 57.3672 141.295 57.3672H137.647C137.57 57.3672 137.493 57.3413 137.416 57.3154C137.339 57.2896 137.287 57.2378 137.236 57.186C137.185 57.1343 137.159 57.0566 137.133 56.979C137.107 56.9014 137.133 56.7979 137.185 56.6685L143.351 42.2539ZM144.584 51.5444H146.46C146.588 51.5444 146.691 51.4927 146.768 51.415C146.819 51.3115 146.819 51.208 146.768 51.0786C146.64 50.7163 146.46 50.3022 146.254 49.8364C146.049 49.3706 145.869 48.9048 145.715 48.4907C145.663 48.3613 145.586 48.3096 145.535 48.3096C145.483 48.3096 145.406 48.3613 145.355 48.4907L144.302 51.0786C144.25 51.208 144.25 51.3115 144.302 51.415C144.327 51.4927 144.43 51.5444 144.584 51.5444Z" fill="#302D2D"/><path d="M164.085 57.0054L159.974 52.373C159.871 52.2695 159.794 52.2178 159.743 52.2437C159.692 52.2695 159.64 52.3472 159.64 52.5024V56.876C159.64 57.0312 159.589 57.1348 159.486 57.2383C159.383 57.3418 159.255 57.3936 159.126 57.3936H155.94C155.786 57.3936 155.658 57.3418 155.581 57.2383C155.478 57.1348 155.452 57.0054 155.452 56.876V42.2803C155.452 42.1509 155.504 42.0215 155.581 41.918C155.683 41.8145 155.786 41.7627 155.94 41.7627H159.126C159.255 41.7627 159.383 41.8145 159.486 41.918C159.589 42.0215 159.64 42.1509 159.64 42.2803V47.1973C159.64 47.3525 159.666 47.4302 159.717 47.4302C159.769 47.4561 159.846 47.4043 159.923 47.3008L163.725 42.2026C163.802 42.0991 163.931 41.9956 164.085 41.918C164.239 41.8403 164.393 41.7886 164.548 41.7886H168.581C168.736 41.7886 168.838 41.8662 168.89 41.9956C168.941 42.125 168.915 42.2544 168.838 42.3579L163.289 49.397C163.211 49.5005 163.16 49.6299 163.16 49.7852C163.16 49.9404 163.212 50.0698 163.314 50.1475C163.802 50.665 164.291 51.2085 164.83 51.7778C165.37 52.3472 165.909 52.9165 166.423 53.4858C166.963 54.0552 167.477 54.6245 167.99 55.1938C168.504 55.7632 168.992 56.2808 169.455 56.7725C169.558 56.876 169.583 57.0054 169.558 57.1606C169.532 57.29 169.429 57.3677 169.275 57.3677H164.882C164.727 57.3677 164.573 57.3418 164.393 57.2642C164.291 57.2124 164.188 57.1089 164.085 57.0054Z" fill="#302D2D"/><path d="M175.133 46.2139V47.1196C175.133 47.249 175.184 47.3784 175.287 47.4819C175.39 47.5854 175.493 47.6372 175.647 47.6372H181.376C181.505 47.6372 181.633 47.689 181.736 47.7925C181.839 47.896 181.89 48.0254 181.89 48.1548V51.105C181.89 51.2344 181.839 51.3638 181.736 51.4673C181.633 51.5708 181.505 51.6226 181.376 51.6226H175.647C175.493 51.6226 175.39 51.6743 175.287 51.7778C175.184 51.8813 175.133 52.0107 175.133 52.1401V52.9424C175.133 53.0977 175.184 53.2012 175.287 53.3047C175.39 53.4082 175.493 53.46 175.647 53.46H182.224C182.353 53.46 182.481 53.5117 182.584 53.6152C182.687 53.7187 182.738 53.8481 182.738 53.9775V56.9277C182.738 57.083 182.687 57.1865 182.584 57.29C182.481 57.3936 182.353 57.4453 182.224 57.4453H171.459C171.33 57.4453 171.202 57.3936 171.099 57.29C170.996 57.1865 170.945 57.0571 170.945 56.9277V42.2803C170.945 42.1509 170.996 42.0215 171.099 41.918C171.202 41.8145 171.33 41.7627 171.459 41.7627H182.224C182.353 41.7627 182.481 41.8145 182.584 41.918C182.687 42.0215 182.738 42.1509 182.738 42.2803V45.1787C182.738 45.334 182.687 45.4375 182.584 45.541C182.481 45.6445 182.353 45.6963 182.224 45.6963H175.647C175.493 45.6963 175.39 45.748 175.287 45.8516C175.184 45.9551 175.133 46.0586 175.133 46.2139Z" fill="#302D2D"/></svg>
  <svg x="803" y="764" width="74" height="17" viewBox="0 0 224 51.3"><path d="M219.858 12.0673C221.827 14.267 222.7 17.0055 223.325 19.7648C226.537 33.9756 217.996 47.6386 203.764 50.6969C188.21 54.0394 174.997 43.177 173.013 29.5702C172.134 23.9752 173.141 18.2467 175.876 13.2871C176.128 12.8222 176.293 12.2657 176.877 12.0584L184.222 12.1798L181.717 17.4763C181.717 17.6579 181.717 17.8394 181.717 18.021C183.526 23.2701 186.898 27.5837 190.098 31.9979C190.442 33.8719 190.187 35.8466 191.022 37.6378C193.355 40.2727 195.687 42.9096 198.018 45.5484C198.19 45.7438 198.397 45.8682 198.61 45.5781C199.676 44.9267 199.987 43.7307 200.576 42.7566C201.914 40.554 203.368 38.4105 204.425 36.045L204.75 35.5299C206.059 35.0206 206.885 33.8483 208.043 33.1348C208.43 32.9221 208.746 32.6001 208.952 32.209C209.158 31.8179 209.244 31.3748 209.2 30.935C209.132 28.7235 209.176 26.4942 209.173 24.2915C209.698 23.7726 210.192 23.224 210.654 22.6484C211.782 21.0645 213.558 19.9839 214.221 18.0477C214.338 18.0629 214.457 18.0429 214.563 17.9901C214.668 17.9374 214.755 17.8543 214.813 17.7516C215.281 17.0411 214.778 16.446 214.574 15.8213C214.183 14.6193 213.434 13.5357 213.357 12.2272L219.858 12.0673Z" fill="#287ABE"/><path d="M184.225 12.1798L176.88 12.0584C177.848 9.53897 179.763 7.73598 181.741 6.07806C185.145 3.22111 189.05 1.24936 193.459 0.452969C201.127 -0.911856 208.102 0.749027 214.275 5.52739C216.575 7.30374 218.662 9.31693 219.864 12.0584L213.351 12.2005L208.022 12.1354V10.3383L211.841 10.3176L211.965 10.2495L211.844 10.0482C211.201 9.41788 210.433 8.92919 209.589 8.61357C208.746 8.29794 207.846 8.16238 206.947 8.2156C201.695 8.33994 196.44 8.26593 191.188 8.25408C189.346 8.25408 187.813 9.0416 186.336 10.0156C184.974 10.0748 184.95 11.2383 184.598 12.1265L184.225 12.1798Z" fill="#B8E1F2"/><path d="M181.717 18.0151C181.717 17.8335 181.717 17.6519 181.717 17.4704C184.787 16.5585 187.606 15.0427 190.495 13.7104C191.41 13.2871 192.671 13.3226 193.16 12.1502C194.918 12.2242 196.712 11.8541 198.438 12.3811C198.452 16.5259 198.466 20.6806 198.48 24.8451H198.208C194.835 23.4033 191.487 21.9053 188.085 20.5405C185.974 19.7026 184.041 18.3644 181.717 18.0151Z" fill="#B6E0F2"/><path d="M181.717 18.0151C184.038 18.3615 185.971 19.6967 188.094 20.5494C191.496 21.9142 194.844 23.4122 198.216 24.854C197.947 26.1182 196.695 26.4202 195.907 27.1633C194.11 28.8626 191.961 30.1564 190.247 31.9594C190.2 31.9806 190.15 31.9927 190.099 31.9949C186.901 27.5807 183.529 23.2701 181.717 18.0151Z" fill="#90D4F0"/><path d="M209.176 24.2797C209.176 26.4942 209.135 28.7205 209.203 30.9232C209.247 31.363 209.161 31.806 208.955 32.1971C208.749 32.5883 208.433 32.9102 208.046 33.1229C206.888 33.8364 206.062 35.0088 204.753 35.518C203.356 33.7417 202.486 31.6219 201.233 29.7567C200.381 28.4896 200.028 26.8554 198.569 26.0264C198.569 25.7461 198.574 25.4659 198.584 25.1856L199.149 24.8895C199.404 25.2034 199.762 25.1619 200.093 25.1323C203.115 24.856 206.143 24.5718 209.176 24.2797Z" fill="#4D8AC8"/><path d="M191.022 37.6407C190.187 35.8466 190.43 33.8749 190.101 31.9979C190.153 31.9956 190.203 31.9836 190.249 31.9624C192.914 32.0749 195.578 32.143 198.225 32.5545C198.225 34.6269 198.22 36.6993 198.21 38.7717C197.676 38.751 197.144 38.6936 196.618 38.6C194.747 38.3098 192.941 37.6467 191.022 37.6407Z" fill="#B6E0F2"/><path d="M198.208 38.7687C198.208 36.6963 198.212 34.6239 198.222 32.5515C198.835 30.4021 198.317 28.1936 198.557 26.0383C200.019 26.8672 200.375 28.5044 201.221 29.7686C202.477 31.6367 203.344 33.7446 204.742 35.5299L204.416 36.045L198.462 38.7835C198.377 38.787 198.292 38.7821 198.208 38.7687Z" fill="#58AFDD"/><path d="M208.007 12.1443L213.336 12.2094C213.413 13.518 214.162 14.6016 214.553 15.8036C214.757 16.4312 215.26 17.0233 214.793 17.7338C214.676 17.7245 214.56 17.7471 214.455 17.7993C214.351 17.8515 214.263 17.9312 214.2 18.0299C213.09 18.2371 212.128 18.8263 211.124 19.2763C210.121 19.7263 209.23 20.4783 208.042 20.4457C208.042 17.6983 208.038 14.9519 208.028 12.2065L208.007 12.1443Z" fill="#4D8AC8"/><path d="M208.048 20.4487C209.233 20.4813 210.121 19.7263 211.13 19.2793C212.14 18.8322 213.096 18.2401 214.206 18.0329C213.54 19.972 211.767 21.0497 210.639 22.6336C210.177 23.2092 209.683 23.7578 209.159 24.2767C206.129 24.5629 203.099 24.8481 200.07 25.1323C199.738 25.1619 199.38 25.2034 199.125 24.8896C201.218 23.9214 203.326 22.9859 205.396 21.9704C206.313 21.5263 207.376 21.3161 208.048 20.4487Z" fill="#266FA8"/><path d="M198.208 38.7687C198.295 38.7778 198.383 38.7778 198.471 38.7687C198.518 41.0346 198.565 43.3043 198.61 45.578C198.397 45.8741 198.193 45.7409 198.018 45.5484C195.683 42.9175 193.351 40.2845 191.022 37.6496C192.938 37.6496 194.744 38.3187 196.615 38.6089C197.141 38.6985 197.674 38.7519 198.208 38.7687Z" fill="#90D4F0"/><path d="M193.172 12.1561C192.671 13.3404 191.41 13.293 190.507 13.7163C187.618 15.0486 184.799 16.5644 181.729 17.4763L184.237 12.1798L184.604 12.1324L193.172 12.1561Z" fill="#90D4F0"/><path d="M198.607 45.5692C198.562 43.3033 198.515 41.0336 198.468 38.7598L204.422 36.0213C203.365 38.3898 201.911 40.5303 200.573 42.7329C199.999 43.7188 199.673 44.9149 198.607 45.5692Z" fill="#266FA8"/><path d="M214.209 18.0388C214.272 17.9401 214.36 17.8604 214.464 17.8082C214.569 17.756 214.685 17.7334 214.802 17.7427C214.744 17.8454 214.656 17.9285 214.551 17.9812C214.445 18.034 214.326 18.054 214.209 18.0388Z" fill="#266FA8"/><path d="M186.327 10.0215C187.807 9.04752 189.338 8.2452 191.179 8.26001C196.434 8.26001 201.689 8.34586 206.938 8.22152C207.838 8.16785 208.74 8.30347 209.584 8.61963C210.429 8.93579 211.197 9.42547 211.841 10.0571C211.841 10.1479 211.841 10.2387 211.841 10.3294L207.992 10.3502H188.186C187.547 10.3502 186.901 10.3887 186.327 10.0215Z" fill="#F4FAFD"/><path d="M186.327 10.0215C186.901 10.3887 187.547 10.3472 188.186 10.3472H207.992V12.1443L208.016 12.2094L198.45 12.39C196.709 11.866 194.93 12.2331 193.172 12.1591L184.586 12.1384C184.944 11.2443 184.965 10.0808 186.327 10.0215Z" fill="#E9F6FD"/><path d="M211.841 10.3176C211.841 10.2268 211.841 10.136 211.841 10.0452L211.962 10.2465L211.841 10.3176Z" fill="#E9F6FD"/><path d="M198.45 12.39L208.034 12.2153C208.034 14.9627 208.039 17.7092 208.048 20.4546C207.376 21.3221 206.316 21.5323 205.408 21.9793C203.335 22.9948 201.23 23.9303 199.137 24.8984L198.572 25.1945L198.492 24.8599L198.45 12.39Z" fill="#68C1EA"/><path d="M5.53331 39.1625H0V12.6328H5.53331V39.1625Z" fill="#2C7CBF"/><path d="M10.2703 25.8961C10.2703 17.7871 16.1914 12.1384 24.1021 12.1384C29.1054 12.1384 33.1585 14.3736 35.4322 17.4674L30.6538 20.5908C29.2505 18.6575 26.9028 17.4467 24.0961 17.4467C19.3592 17.4467 16.0227 20.8957 16.0227 25.8991C16.0227 30.9025 19.3207 34.3515 24.0577 34.3515C26.9383 34.3515 29.4755 33.0637 30.9173 30.6745L35.5417 33.9311C33.117 37.5697 28.8331 39.6539 24.0577 39.6539C16.147 39.6539 10.2703 34.0081 10.2703 25.8961Z" fill="#2C7CBF"/><path d="M60.1411 34.0437V39.1595H40.2816V12.6328H59.8362V17.7486H45.8149V23.1665H59.2293V28.3238H45.8149V34.0466L60.1411 34.0437Z" fill="#2C7CBF"/><path d="M87.2007 31.6574C87.2007 35.9384 84.0181 39.1625 78.8637 39.1625H64.875V12.6328H77.9874C82.9907 12.6328 86.0224 15.5519 86.0224 19.7559C86.0224 22.5211 84.6191 24.6053 82.1559 25.4787C85.3414 26.0471 87.2007 28.3238 87.2007 31.6574ZM70.4083 17.3312V23.3589H77.0015C79.1627 23.3589 80.2581 22.1747 80.2581 20.1763C80.2581 18.4 79.1598 17.3342 77.0015 17.3342L70.4083 17.3312ZM81.4364 31.163C81.4364 29.1173 80.2522 27.9804 77.9489 27.9804H70.4083V34.4611H77.9489C80.2611 34.4611 81.4364 33.2502 81.4364 31.166V31.163Z" fill="#2C7CBF"/><path d="M111.791 34.0437V39.1595H91.9376V12.6328H111.477V17.7486H97.465V23.1665H110.885V28.3238H97.465V34.0466L111.791 34.0437Z" fill="#2C7CBF"/><path d="M125.623 29.4962H122.07V39.1595H116.528V12.6328H128.391C134.114 12.6328 137.602 16.0818 137.602 21.0467C137.602 25.0257 135.366 28.0574 131.5 29.1173L138.398 39.1595H131.766L125.623 29.4962ZM122.07 24.3418H128.249C130.635 24.3418 131.888 23.205 131.888 21.0852C131.888 18.9654 130.638 17.7871 128.249 17.7871H122.059L122.07 24.3418Z" fill="#2C7CBF"/><path d="M154.613 39.6539C146.694 39.6539 140.781 34.0081 140.781 25.8606C140.781 17.7131 146.806 12.1413 154.729 12.1413C158.873 12.1413 162.648 13.6571 165.387 16.7272L161.094 20.3273C160.312 19.3995 159.331 18.659 158.225 18.1605C157.119 17.6619 155.915 17.418 154.702 17.4467C149.814 17.4467 146.554 20.9342 146.554 25.8991C146.554 30.864 149.775 34.3515 154.74 34.3515C158.379 34.3515 160.958 32.4183 161.828 28.9692H154.702V24.3803H167.853C167.949 25.02 167.987 25.6671 167.965 26.3136C168.024 34.1976 162.42 39.6539 154.613 39.6539Z" fill="#2C7CBF"/></svg>
  <text x="1058" y="781" class="meta" font-size="14">Any Cloud</text>
  <svg x="1132" y="766" width="15" height="15" viewBox="0 0 16 16"><path d="M3.39373 5.58639C3.91293 3.52534 5.77786 2 8 2C10.5504 2 12.6314 4.01005 12.7451 6.5324C14.1591 6.7189 15.3247 7.69323 15.7866 9H14.1211C13.7175 8.39701 13.0301 8 12.25 8H12C11.5858 8 11.25 7.66421 11.25 7.25V6.75C11.25 4.95507 9.79493 3.5 8 3.5C6.34131 3.5 4.97186 4.74324 4.7745 6.34833C4.73041 6.70685 4.43704 6.98301 4.07651 7.00536C2.63892 7.09448 1.5 8.28952 1.5 9.75C1.5 11.1845 2.59873 12.3629 4 12.4888V14C3.93845 14 3.87864 13.9926 3.8214 13.9786C1.67511 13.7633 0 11.9526 0 9.75C0 7.69604 1.45669 5.98279 3.39373 5.58639Z" fill="#2D6FC7"/><path fill-rule="evenodd" clip-rule="evenodd" d="M8 7C9.24264 7 10.25 8.00736 10.25 9.25C10.25 9.59736 10.1713 9.92634 10.0307 10.2201L10.5299 10.7193C10.8237 10.5787 11.1526 10.5 11.5 10.5C12.7426 10.5 13.75 11.5074 13.75 12.75C13.75 13.9926 12.7426 15 11.5 15C10.2574 15 9.25 13.9926 9.25 12.75C9.25 12.4026 9.32871 12.0737 9.46928 11.7799L8.97006 11.2807C8.67634 11.4213 8.34736 11.5 8 11.5C6.75736 11.5 5.75 10.4926 5.75 9.25C5.75 8.00736 6.75736 7 8 7ZM8.75 9.25C8.75 8.83579 8.41421 8.5 8 8.5C7.58579 8.5 7.25 8.83579 7.25 9.25C7.25 9.66421 7.58579 10 8 10C8.41421 10 8.75 9.66421 8.75 9.25ZM12.25 12.75C12.25 12.3358 11.9142 12 11.5 12C11.0858 12 10.75 12.3358 10.75 12.75C10.75 13.1642 11.0858 13.5 11.5 13.5C11.9142 13.5 12.25 13.1642 12.25 12.75Z" fill="#2D6FC7"/></svg>
  <text x="1174" y="781" class="meta" font-size="14">Any Model</text>
  <svg x="1252" y="766" width="15" height="15" viewBox="0 0 100 100"><g stroke="#2D6FC7" stroke-width="9" stroke-linecap="round"><line x1="30" y1="36" x2="68" y2="22"/><line x1="30" y1="36" x2="73" y2="58"/><line x1="30" y1="36" x2="43" y2="74"/><line x1="68" y1="22" x2="73" y2="58"/><line x1="73" y1="58" x2="43" y2="74"/></g><circle cx="30" cy="36" r="13" fill="#2D6FC7"/><circle cx="68" cy="22" r="13" fill="#2D6FC7"/><circle cx="73" cy="58" r="13" fill="#2D6FC7"/><circle cx="43" cy="74" r="13" fill="#2D6FC7"/><circle cx="30" cy="36" r="6" fill="#EBF1F6"/><circle cx="68" cy="22" r="6" fill="#EBF1F6"/><circle cx="73" cy="58" r="6" fill="#EBF1F6"/><circle cx="43" cy="74" r="6" fill="#EBF1F6"/></svg>
  <text x="1290" y="781" class="meta" font-size="14">Any Data</text>
  <svg x="1360" y="766" width="15" height="15" viewBox="0 0 16 16"><path fill-rule="evenodd" clip-rule="evenodd" d="M8.64576 0.368453C8.51082 0.140099 8.2653 0 8.00006 0C7.73482 0 7.4893 0.140099 7.35437 0.368453L4.10437 5.86845C3.96736 6.1003 3.96518 6.38781 4.09865 6.62171C4.23213 6.85561 4.48076 7 4.75006 7H11.2501C11.5194 7 11.768 6.85561 11.9015 6.62171C12.0349 6.38781 12.0328 6.1003 11.8958 5.86845L8.64576 0.368453ZM8.00006 2.22426L9.93572 5.5H6.0644L8.00006 2.22426ZM8.5 9.25C8.5 8.83579 8.83579 8.5 9.25 8.5H14.25C14.6642 8.5 15 8.83579 15 9.25V14.25C15 14.6642 14.6642 15 14.25 15H9.25C8.83579 15 8.5 14.6642 8.5 14.25V9.25ZM10 10V13.5H13.5V10H10ZM4.25 10C3.2835 10 2.5 10.7835 2.5 11.75C2.5 12.7165 3.2835 13.5 4.25 13.5C5.2165 13.5 6 12.7165 6 11.75C6 10.7835 5.2165 10 4.25 10ZM1 11.75C1 9.95507 2.45507 8.5 4.25 8.5C6.04493 8.5 7.5 9.95507 7.5 11.75C7.5 13.5449 6.04493 15 4.25 15C2.45507 15 1 13.5449 1 11.75Z" fill="#2D6FC7"/></svg>
</svg>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
What is Lakebase, and why does it matter for feature engineering?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #1B3139;">

<p><strong>Lakebase</strong> is Databricks' fully managed Postgres database, built into the Databricks platform. It separates Postgres compute from durable storage, so compute <strong>autoscales</strong> with demand and is <strong>suspended when inactive</strong> to save cost. It also supports <strong>branching</strong> and <strong>automatic failover</strong>.</p>

<p>Delta tables are optimized for scanning many rows, such as building a training set or scoring a whole table in batch. Lakebase is optimized for the opposite pattern: looking up a <strong>few rows by primary key</strong>, very quickly. That is exactly what a model needs at inference time.</p>

<ul style="margin: 12px 0; padding-left: 24px;">
<li><strong>Powers Online Feature Stores</strong> — When you publish a feature table, Databricks copies it into a Lakebase-backed Online Feature Store. Real-time Model Serving and Feature Serving endpoints read features from there with low latency.</li>
<li><strong>Synced from Unity Catalog</strong> — The Delta feature table stays the source of truth (the offline store). The online copy is kept up to date by a one-time snapshot, by scheduled or on-demand incremental syncs, or by continuous streaming.</li>
<li><strong>Same features in training and serving</strong> — The model trains on the offline Delta table and is served from the synced online copy of the <em>same</em> table, so the feature values match. This prevents train/serve skew.</li>
<li><strong>Governed by Unity Catalog</strong> — Online stores and their tables are Unity Catalog objects. A model logged with <code>fe.log_model</code> uses Unity Catalog to find its features in the online store automatically.</li>
<li><strong>Safe to experiment</strong> — Branches let you test changes in isolation, and you can create a branch from any point in your history window to recover from mistakes.</li>
</ul>

<p>With that foundation in place, the next section shows how to publish a feature table to an <strong>Online Feature Store</strong> and serve it in real time.</p>

</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;"><strong style="color: #0d47a1;">Learn More</strong><p style="margin: 8px 0 0 0; color: #333;">Lakebase (<a href="https://docs.databricks.com/aws/en/oltp/" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/oltp/" style="color: #1976d2;">Azure</a>)</p></div>

### B8. Online Feature Stores and Feature Serving

Feature tables in Unity Catalog are the **offline store**: Delta tables that are optimized for training and batch scoring. Real-time applications, such as a model that scores a transaction while the customer waits, need feature lookups in milliseconds. For this, you publish feature tables to an **Online Feature Store**, which is backed by **Lakebase** (introduced in B7), the Databricks managed Postgres service.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Store</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Backed by</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Used for</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Offline store</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Delta tables in Unity Catalog</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Training sets, batch scoring, and analysis</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Online Feature Store</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Lakebase (managed Postgres)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Low-latency lookups for real-time Model Serving endpoints and Feature Serving endpoints</td></tr></tbody></table></div>

Publishing keeps the online copy synchronized with the offline feature table, so the same feature definitions are used in training and in production. This is how a feature store prevents **train/serve skew**.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from databricks.feature_engineering import FeatureEngineeringClient</code>
</br>
<code>fe = FeatureEngineeringClient()</code>
</br>
<code># Create an online store (one-time setup)</code>
<code>online_store = fe.create_online_store(name="customer-online-store", capacity="CU_1")</code>
</br>
<code># Publish a feature table to the online store</code>
<code>fe.publish_table(</code>
<code>    online_store=online_store,</code>
<code>    source_table_name="catalog.schema.customer_features",</code>
<code>    online_table_name="catalog.schema.customer_features_online",</code>
<code>)</code></pre>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Online Feature Stores replace the legacy standalone <b>online tables</b>. Publishing requires a primary key on the source table and, for the default incremental sync (<code>TRIGGERED</code> mode), Change Data Feed enabled on it (<code>delta.enableChangeDataFeed = true</code>). Online stores incur compute cost, so this course describes the concept but does not create one in the labs. A model deployed to Model Serving that was logged with feature metadata looks up its features from the online store automatically, as described in B4.</div></div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;"><strong style="color: #0d47a1;">Learn More</strong><p style="margin: 8px 0 0 0; color: #333;">Online Feature Stores (<a href="https://docs.databricks.com/aws/en/machine-learning/feature-store/online-feature-store" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/online-feature-store" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/machine-learning/feature-store/online-feature-store" style="color: #1976d2;">GCP</a>)</p></div>

### B9. Feature Views: Declaring Features Instead of Building Tables

So far in this lecture, you have worked with **feature tables**: you write the code that computes the features, and you own the job that keeps the table up to date. Feature Engineering in Unity Catalog also offers a second way to author features, called **Feature Views** (Public Preview).

With a Feature View, you **declare** what a feature is (the source table, the entity key, the time column, and the calculation, such as a 7-day sum) and register it in Unity Catalog. **Databricks then creates and manages the pipelines** that compute the feature, keep it fresh, and make it available for training and serving.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;"></th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Feature tables</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Feature Views (Public Preview)</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">What you create</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A Delta table in Unity Catalog with a primary key</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A <code>Feature</code> object in Unity Catalog that describes how to compute the feature</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Who computes the features</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">You do, with your own notebook, job, or pipeline</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Databricks creates and manages the feature pipelines</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Sources</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Anything you can write to a Delta table</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Delta tables, Kafka streams, and request-time data</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Typical calculations</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Any logic you write</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Column selections and time-windowed aggregations, such as Sum, Avg, Count, Min, and Max over tumbling or sliding windows</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Best fit</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Flexible, multi-stage batch workloads</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Most new projects, time-windowed aggregations, and streaming features</td></tr></tbody></table></div>

Databricks recommends Feature Views for most new cases, and feature tables for flexible, multi-stage batch workloads. Both are governed by Unity Catalog and both work with the training workflow you learned in B6: <code>create_training_set</code> builds a point-in-time correct training set, and a model logged with <code>fe.log_model</code> carries its feature metadata for serving.

The following code illustrates the pattern. It is not run in this course.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from datetime import timedelta</code>
<code>from databricks.feature_engineering import FeatureEngineeringClient</code>
<code>from databricks.feature_engineering.entities import (</code>
<code>    DeltaTableSource, AggregationFunction, Sum, SlidingWindow,</code>
<code>)</code>
</br>
<code>fe = FeatureEngineeringClient()</code>
</br>
<code># 1. Point to the source data: a Delta table in Unity Catalog</code>
<code>source = DeltaTableSource(</code>
<code>    catalog_name=&quot;catalog&quot;, schema_name=&quot;schema&quot;, table_name=&quot;transactions&quot;,</code>
<code>)</code>
</br>
<code># 2. Declare the feature: total spend over the last 7 days, per user</code>
<code>spend_7d = fe.create_feature(</code>
<code>    source=source,</code>
<code>    function=AggregationFunction(</code>
<code>        Sum(input=&quot;amount&quot;),</code>
<code>        SlidingWindow(window_duration=timedelta(days=7), slide_duration=timedelta(days=1)),  # or RollingWindow for a single trailing window</code>
<code>    ),</code>
<code>    entity=[&quot;user_id&quot;],</code>
<code>    timeseries_column=&quot;transaction_time&quot;,</code>
<code>    catalog_name=&quot;catalog&quot;, schema_name=&quot;schema&quot;,</code>
<code>    name=&quot;spend_7d&quot;,</code>
<code>)</code>
</br>
<code># 3. Build a point-in-time correct training set from labeled data</code>
<code>training_set = fe.create_training_set(</code>
<code>    df=labeled_df, features=[spend_7d], label=&quot;target&quot;,</code>
<code>)</code></pre>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Feature Views are in <b>Public Preview</b>, and a workspace admin may need to enable them from the <b>Previews</b> page. They require serverless compute or Databricks Runtime 17.0 ML or above, and <code>databricks-feature-engineering</code> 0.16.0 or above. The demos and labs in this course use feature tables, which are generally available.</div></div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;"><strong style="color: #0d47a1;">Learn More</strong><p style="margin: 8px 0 0 0; color: #333;">Feature Views (<a href="https://docs.databricks.com/aws/en/machine-learning/feature-store/feature-views" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/feature-views" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/machine-learning/feature-store/feature-views" style="color: #1976d2;">GCP</a>)</p></div>

## C. Conclusion

In this lecture, you covered the foundational concepts and architecture of Feature Engineering in Unity Catalog:

1. **Why a Feature Store** — solves discovery, lineage, train/serve skew, and online serving through reusable feature logic and Delta-backed feature tables.
2. **Feature Tables = UC Tables** — any Delta table in Unity Catalog with a primary key can be used as a feature table, and it is governed like any other Unity Catalog table: lineage, permissions, time travel, and cross-workspace access.
3. **Unified Governance** — Feature Engineering in Unity Catalog uses UC's `GRANT`/`REVOKE` permission model and lineage graph, so all UC security rules and future improvements apply automatically to feature data.
4. **MLflow and Model Serving Integration** — a model logged with `fe.log_model` carries its feature lookup metadata, enabling automatic feature lookup at inference time in both batch (offline store) and online (real-time) modes.
5. **Workspace FS vs. UC FS** — new work should use `FeatureEngineeringClient` (UC FS), which replaces the legacy Workspace Feature Store with cross-workspace sharing, unified lineage, and robust governance.
6. **Point-in-Time Lookups** — time series feature tables plus `timestamp_lookup_key` return the feature values known at each label's timestamp, preventing future-data leakage.
7. **Lakebase** — Databricks' managed Postgres database, built for fast primary-key lookups. It complements Delta feature tables (the offline store) and is the database behind Online Feature Stores.
8. **Online Feature Stores** — publishing feature tables to a Lakebase-backed online store provides millisecond lookups for real-time serving with the same feature definitions used in training.
9. **Feature Views (Public Preview)** — declare a feature (source, entity, time column, and calculation) and Databricks manages the pipelines that compute it. Recommended for most new cases; feature tables remain the best fit for flexible, multi-stage batch workloads.



&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>